# Week 7 Lab. Differencing, ARIMA and SARIMA on One Real Series
**Time Series Analysis & Random Processes** · Graduate School of Data Science, Chonnam National University · notebook v6 (2026-10-13)

---

### 이 실습에서 하는 일 / What this lab does

오늘 덱의 세 질문을 실제 계열 하나, 미국 여가·숙박 업종의 월별 고용(FPP-py §9.9)에 직접 돌려 봅니다. 순서는 덱 「Five Steps From Raw Data to a Checked Forecast」 의 다섯 단계 그대로입니다. 여기서 찍히는 숫자가 슬라이드의 숫자와 같고, 「IN THE LAB」 표시가 있는 장마다 그 표시대로 해 보는 셀이 있습니다.
*This notebook runs today's three questions on one real series, the monthly US jobs in leisure and hospitality (FPP-py §9.9), in the order of the deck's "Five Steps From Raw Data to a Checked Forecast". The numbers you print are the ones on the slides, and every slide marked **IN THE LAB** has a cell here that does what its badge says.*

1. **차분을 몇 번 해야 하나? / How many differences does this series need?**
2. **차분한 뒤 남은 것을 어떻게 모형으로 만드나? / After differencing, how do we model what is left?**
3. **예측이 실제로 맞는지 어떻게 아나? / How do we know the forecast actually works?**

| 단계 / Step | 먼저 예측 / Before running, predict | 실행하고 확인 / Then run and check | 짝 슬라이드 / Matching slides |
|---|---|---|---|
| 1 | 그림과 ACF 를 보고, 차분이 필요할까?<br>Plot the series and its ACF. Does it need differencing? | ADF 와 KPSS 를 돌려 결정표의 줄을 찾는다<br>Run ADF and KPSS. Find your row in the decision table. | 「Can We Read This ACF the Way We Did Last Week?」 · 「Read the Two Tests Together」 · 「Can the Tests Tell Series A From Series B?」 |
| 2 | 어떤 차분이 정상으로 만들까?<br>Which differences will make it stationary? | 차분하고 다시 검정한다<br>Difference, then test again. | 「What If the Pattern Repeats Every 12 Months?」 · 「How Many Differences Does the Jobs Series Need?」 |
| 3 | ACF 와 PACF 를 읽고 후보 둘을 쓴다<br>Read the ACF and PACF. Write two candidates. | 둘 다 맞추고 AIC · BIC 를 비교한다<br>Fit both. Compare AIC and BIC. | 「What Memory Is Left After Both Differences?」 · 「One Name for Two Kinds of Memory」 · 「Which Candidates Are Worth Checking?」 |
| 4 | 잔차에 무늬가 남을까?<br>Will the residuals still contain a pattern? | 시차 24개로 Ljung–Box 를 돌린다<br>Run Ljung–Box on 24 lags. | 「Did the Model Leave a Pattern Behind?」 |
| 5 | 마지막 24개월에서 어느 예측이 이길까?<br>Which forecast will win on the last 24 months? | 계절 나이브 예측과 MAE · MASE 로 견준다<br>Compare with the seasonal naive forecast by MAE and MASE. | 「What Should an Error Measure Tell Us?」 · 「Did SARIMA Beat the Seasonal Naive Forecast?」 |
| 6 | 선택<br>Optional | 주가 · rolling origin · 3년 예측<br>A stock price, a rolling origin, three years ahead | 「What Do the Tests Say About a Stock Price?」 · 「Three Years of Forecasts for the Jobs Series」 |

1–5 단계는 덱 「In the Lab, Run All Five Steps on One Series」 의 표와 같습니다.
*Steps 1 to 5 are the table on "In the Lab, Run All Five Steps on One Series".*

실제 분석에서는 시험 달을 **먼저** 떼어 두고 1~4단계를 훈련 달로만 합니다. 이 노트북의 1~4절은 덱의 숫자를 그대로 재현하려고 225개월 전체로 돌리는 **전체 표본 시연**이고, 시험 달은 5절에서야 나눕니다. 그래서 3 · 4절의 AIC · BIC · Ljung–Box 는 시험 달까지 본 숫자이고, 예측 성능을 재는 곳은 5절뿐입니다. 후보를 훈련 달만으로 다시 골라 보는 **훈련 달 확인**은 5절의 Try it 과 선택 D 입니다.
*In a real analysis the test months are set aside **first**, and steps 1 to 4 use the training months only. Sections 1 to 4 here are a **full-sample demonstration**: they run on all 225 months so that the deck's numbers come out exactly, and the test months are split off only in section 5. So the AIC, BIC and Ljung–Box values of sections 3 and 4 have seen the test months, and section 5 is the only place that measures forecast accuracy. The **training-only check**, choosing the candidates again from the training months, is the Try it in section 5 and Optional D.*

### 쓰는 법 / How to use it

**`TODO` 가 네 개 있습니다.** 이 네 줄을 직접 채우는 것이 이 노트북의 핵심입니다.
*Four cells are marked **`TODO`**. Those four lines are the part worth doing by hand.*

> 각 `TODO` 는 `… = None` 한 줄입니다. `None` 자리에 코드를 넣으면 되고, 지울 줄은 없습니다. 채우기 전에 실행하면 `NotImplementedError` 와 안내문이 뜨며 멈춥니다. **고장이 아니라 정상입니다.** `TODO` 바로 아래의 **자가검증** 셀이 내 답을 검사하고, 그 아래에 접힌 **Solution (정답)** 셀이 있습니다. 먼저 해 본 뒤에만 여세요. 정답 셀을 돌렸다면 자가검증 셀을 다시 실행하세요.
> *Each `TODO` is one line that reads `… = None`. Replace the `None` with your code; nothing needs deleting. If you run the cell before that, it stops with a `NotImplementedError` and a message. **That is expected, not a broken notebook.** The **self-check** cell right below each `TODO` tests your answer, and below it is a collapsed **Solution (정답)** cell. Open it only after you have tried. If you ran the Solution cell, run the self-check again.*

> ### 먼저 **파일 > Drive에 사본 저장** / First, **File > Save a copy in Drive**
> 올려 둔 링크는 **읽기 전용**으로 열립니다. 사본을 먼저 저장하지 않으면 입력한 내용이 탭을 닫을 때 사라집니다.
> *The posted link opens **read-only**. Save your own copy first, or everything you type disappears when you close the tab.*
>
> **제출하지 않습니다.** 7주차 실습 시간(약 35분)용입니다. 시간 안에 다 못 하면 4 · 5단계는 집에서 이어서 하세요. 오늘 내용은 8주차 중간고사 범위입니다.
> *Nothing to submit. This is the Week 7 lab block (about 35 minutes). If time runs out, finish steps 4 and 5 at home. Today's topics are in the Week 8 midterm.*

---
## 0. 준비 / Setup

설치할 것은 없습니다. 아래에 쓰는 NumPy · pandas · Matplotlib · statsmodels 는 전부 Colab 에 이미 들어 있고, 자료도 노트북 안에 있어 내려받을 것이 없습니다.
*Nothing to install. NumPy, pandas, Matplotlib and statsmodels all ship with Colab, and the data are inside the notebook, so nothing is downloaded.*

**위에서 아래로 차례로 실행하세요.** 3 · 4 · 5절은 앞 절이 만든 계열과 적합 결과를 씁니다. 런타임을 다시 시작했다면 준비 셀과 자료 셀부터 다시 실행하세요. 무작위 숫자는 1절의 5주차 계열(시드 101) 말고는 쓰지 않습니다.
*Run the cells from top to bottom. Sections 3, 4 and 5 use the series and fits made by earlier sections. After a runtime restart, run the setup and data cells again first. Nothing is random except the Week 5 series in section 1 (seed 101).*

그림의 축 이름은 영문으로 둡니다. 한글 축을 쓰려면 바로 아래 선택 셀을 실행하세요.
*Plot labels are kept in English. Run the optional cell below if you want Korean labels.*

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tools.sm_exceptions import InterpolationWarning

np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (11, 3.4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
GREEN, GOLD, RED = "#1E4A32", "#D9AE3A", "#B34F3F"          # 덱과 같은 색 / the deck's colours

# 빈칸(TODO)을 아직 안 채웠을 때 보여 줄 안내입니다. 고장이 아니라는 뜻입니다.
# Message shown when a TODO is still blank. It means the notebook is fine, not broken.
BLANK_MSG = (
    "\n\n"
    "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
    "     위의 None 자리에 코드를 넣고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
    "     막힐 때 자가검증 셀 아래의 'Solution / 정답' 셀을 실행하고, 자가검증 셀을 다시 실행하면 이어서 진행됩니다.\n\n"
    "  This is expected, not a broken notebook.\n"
    "     Replace None above with your code and re-run. Nothing needs deleting.\n"
    "     If stuck, run the Solution cell below the self-check, then run the self-check again and continue.\n"
)


def test_pair(x, regression="c", kpss_lags=4):
    """ADF 와 KPSS 를 함께 돌려 (ADF 통계량, ADF p, KPSS 통계량, KPSS p) 를 돌려준다.
    Run ADF and KPSS together and return (ADF statistic, ADF p, KPSS statistic, KPSS p)."""
    x = pd.Series(x).dropna().to_numpy(dtype=float)        # 차분으로 생긴 빈칸(NaN)은 뺀다 / drop the NaN left by differencing
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", InterpolationWarning)   # 표 끝의 p 경고. 아래 fmt_p 가 처리 / the table-end warning, handled by fmt_p
        warnings.simplefilter("ignore", FutureWarning)          # 새 statsmodels 의 반환 형식 예고. 숫자는 같다 / a notice about a future return type, same numbers
        adf_stat, adf_p = adfuller(x, autolag="AIC", regression=regression)[:2]
        kpss_stat, kpss_p = kpss(x, regression=regression, nlags=kpss_lags)[:2]
    return adf_stat, adf_p, kpss_stat, kpss_p


def fmt_p(p, test="adf"):
    """p 값을 덱과 같은 꼴로 적는다. 예 / e.g. '= 0.94', '< 0.01', '> 0.10'.
    Write a p-value the way the deck does."""
    if test == "kpss" and p <= 0.01:
        return "< 0.01"     # KPSS 표의 끝. 실제 p 는 이보다 작다 / end of the KPSS table, the true p is smaller
    if test == "kpss" and p >= 0.10:
        return "> 0.10"     # 반대쪽 끝. 실제 p 는 이보다 크다 / the other end, the true p is larger
    if p < 0.001:
        return "< 0.001"
    return f"= {p:.3f}" if p < 0.01 else f"= {p:.2f}"


def mae(e):
    """오차의 절댓값 평균 / mean absolute error."""
    return float(np.mean(np.abs(e)))


def rmse(e):
    """오차 제곱 평균의 제곱근 / root mean squared error."""
    return float(np.sqrt(np.mean(np.square(e))))


def recheck(x, cand, horizon):
    """끝까지 수렴하지 못한 적합을 다른 길로 다시 맞춰, 로그가능도와 예측이 얼마나 움직이는지 돌려준다.
    Refit a fit that did not converge by another route and return how much the log-likelihood and the forecast move."""
    order, seasonal_order = cand
    model = SARIMAX(np.asarray(x, dtype=float), order=order, seasonal_order=seasonal_order)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                              # 점검 중의 경고는 결과로 말한다 / warnings during the check are reported by the result, not printed
        first = model.fit(disp=False)                                # 표에 쓴 것과 같은 적합 / the same fit as in the table
        rough = model.fit(disp=False, method="nm", maxiter=2000)     # 다른 길: 기울기 없이 더듬는 Nelder-Mead / another route: Nelder-Mead, which uses no gradients
        again = model.fit(disp=False, start_params=rough.params)     # 거기서 출발해 기본 방법으로 마무리 / finish with the default method from there
    gap = float(np.max(np.abs(again.forecast(horizon) - first.forecast(horizon))))   # 두 적합의 예측이 가장 크게 다른 달 / the month where the two forecasts differ most
    return first.llf, again.llf, gap


print("준비 완료 / Setup ready")
print("numpy", np.__version__, "· pandas", pd.__version__, "· statsmodels", statsmodels.__version__)

**준비 셀이 만든 도구 / Tools made by the setup cell.** 외울 것은 없습니다. 무엇을 계산하는지만 보세요.
*Nothing to memorise. Just see what each one computes.*

| 이름 / Name | 하는 일 / What it does |
|---|---|
| `test_pair(x)` | ADF 와 KPSS 를 한 번에 돌려 네 숫자(ADF 통계량 · p, KPSS 통계량 · p)를 돌려준다. `regression="ct"` 를 주면 직선 추세를 허용하는 추세판(덱 「Can the Tests Tell Series A From Series B?」)<br>Runs ADF and KPSS and returns four numbers. `regression="ct"` gives the trend versions that allow a straight line |
| `fmt_p(p)` | p 를 덱처럼 「= 0.94」 꼴로 적는다. KPSS 의 p 는 표의 끝(0.01, 0.10)에서 잘리므로 「< 0.01」 · 「> 0.10」 으로<br>Writes p like the deck, as "= 0.94". KPSS p-values are cut at the ends of a table (0.01 and 0.10), so they print as "< 0.01" and "> 0.10" |
| `mae(e)` · `rmse(e)` | 오차 묶음 e 의 MAE · RMSE (덱 「What Should an Error Measure Tell Us?」)<br>The MAE and RMSE of a set of errors e |
| `recheck(x, cand, h)` | 5절에서 「수렴 no」 가 보일 때만 씁니다. 그 후보를 다른 길로 다시 맞춰 (처음 로그가능도, 다시 맞춘 로그가능도, 예측의 최대 차이) 를 돌려준다<br>Used only when section 5 shows "converged: no". Refits that candidate by another route and returns (first log-likelihood, refitted log-likelihood, largest forecast gap) |

KPSS 의 시차는 4 로 둡니다(덱의 숫자와 같은 설정). 다른 값으로 바꾸면 통계량이 달라지지만 결론은 보통 같습니다.
*The KPSS lag setting is 4, the setting behind the deck's numbers. Another value changes the statistic but usually not the conclusion.*

### 선택. 그림에 한글 쓰기 / Optional. Korean labels in plots

Colab 의 기본 글꼴에는 한글이 없어서 한글 축 이름이 네모로 깨집니다. 아래 셀을 한 번 실행하면 고쳐집니다.
*Colab's default font has no Korean glyphs. Run the cell below once if you want Korean axis labels.*

In [ ]:
#@title ▶ 한글 폰트 설치 / Install Korean font (optional) { display-mode: "form" }
# 실행하지 않아도 노트북은 전부 돌아갑니다. 축 이름이 영문이면 이 셀은 건너뛰세요.
# The notebook runs fine without this. Skip it if English labels are fine for you.
import os, subprocess, matplotlib.font_manager as fm

subprocess.run("apt-get -qq -y install fonts-nanum > /dev/null 2>&1", shell=True)
_p = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(_p):
    fm.fontManager.addfont(_p)                       # 캐시를 다시 만들지 않고 바로 등록 / register without rebuilding the cache
    plt.rc("font", family=fm.FontProperties(fname=_p).get_name())
    plt.rc("axes", unicode_minus=False)              # 마이너스 기호 깨짐 방지 / keep the minus sign readable
    print("한글 폰트 준비 완료 / Korean font ready")
else:
    print("폰트를 설치하지 못했습니다. 축 이름은 영문 그대로 두세요. / Could not install the font. Keep the English labels.")

In [ ]:
#@title ▶ 자료 / Data. 미국 여가·숙박 업종 월별 고용, 2001-01 ~ 2019-09 { display-mode: "form" }
# FPP-py §9.9 와 같은 자료입니다(fpp3 패키지의 us_employment, 계열 번호 CEU7000000001).
# 미국 노동통계국(BLS) 고용 통계, 계절조정을 하지 않은 월별 값, 단위는 천 명입니다.
# The same data as FPP-py §9.9 (us_employment in the fpp3 package, series CEU7000000001):
# US Bureau of Labor Statistics, monthly, not seasonally adjusted, in thousands of jobs.
# 교재 패키지의 판이라, 지금 FRED/BLS 에서 받는 판과는 2018-04 이후 최대 0.18백만(1.05%) 다릅니다(연례 벤치마크 수정).
# 덱 「Three Years of Forecasts for the Jobs Series」 의 노트에 적어 두었습니다.
# It is the textbook package's vintage: from 2018-04 on it differs from today's FRED/BLS vintage by up to 0.18 million
# (1.05%), an annual benchmark revision. See the notes of "Three Years of Forecasts for the Jobs Series".
JOBS_THOUSANDS = [
    11328, 11462, 11645, 11936, 12297, 12687, 12757, 12716, 12273, 11945, 11702, 11681,   # 2001
    11385, 11440, 11622, 11823, 12148, 12467, 12536, 12538, 12198, 12013, 11837, 11823,   # 2002
    11568, 11599, 11769, 11984, 12302, 12671, 12776, 12768, 12391, 12177, 12026, 12041,   # 2003
    11760, 11838, 12077, 12355, 12691, 13051, 13128, 13083, 12729, 12519, 12348, 12336,   # 2004
    12064, 12151, 12365, 12711, 13010, 13413, 13514, 13497, 13041, 12782, 12629, 12609,   # 2005
    12339, 12438, 12674, 12960, 13268, 13638, 13790, 13764, 13322, 13125, 12994, 13005,   # 2006
    12703, 12790, 12987, 13260, 13630, 14012, 14092, 14045, 13637, 13441, 13278, 13244,   # 2007
    12905, 12971, 13156, 13396, 13709, 14052, 14129, 14056, 13563, 13300, 13040, 12959,   # 2008
    12607, 12616, 12748, 12977, 13337, 13656, 13746, 13675, 13245, 12956, 12735, 12631,   # 2009
    12315, 12359, 12578, 12915, 13243, 13616, 13720, 13712, 13351, 13085, 12872, 12818,   # 2010
    12477, 12614, 12881, 13233, 13535, 13947, 14032, 14023, 13629, 13418, 13249, 13195,   # 2011
    12904, 13038, 13334, 13626, 13959, 14329, 14411, 14434, 14055, 13838, 13646, 13646,   # 2012
    13324, 13470, 13740, 14074, 14459, 14881, 14978, 14961, 14518, 14346, 14174, 14124,   # 2013
    13815, 13908, 14181, 14524, 14935, 15333, 15407, 15365, 14948, 14787, 14587, 14563,   # 2014
    14234, 14374, 14599, 14947, 15365, 15768, 15878, 15845, 15441, 15293, 15103, 15077,   # 2015
    14747, 14889, 15143, 15480, 15845, 16320, 16421, 16383, 15930, 15721, 15550, 15489,   # 2016
    15124, 15283, 15524, 15887, 16285, 16758, 16886, 16808, 16226, 16098, 15875, 15859,   # 2017
    15437, 15588, 15818, 16132, 16552, 17051, 17164, 17096, 16546, 16402, 16182, 16213,   # 2018
    15880, 15997, 16260, 16575, 16961, 17403, 17463, 17380, 16872,   # 2019 (1-9월 / Jan-Sep)
]
months = pd.period_range("2001-01", periods=len(JOBS_THOUSANDS), freq="M")   # 2001년 1월부터 한 달씩 / one entry per month from January 2001
jobs = pd.Series(np.array(JOBS_THOUSANDS) / 1000, index=months, name="jobs")  # 백만 명 단위, 덱과 같다 / in millions, as on the slides

print("개월 수 / months :", len(jobs))
print("처음 / first :", jobs.index[0], jobs.iloc[0], "   마지막 / last :", jobs.index[-1], jobs.iloc[-1])
# 기대값 / Expected
#   개월 수 / months : 225
#   처음 / first : 2001-01 11.328    마지막 / last : 2019-09 16.872

---
## 1. 차분이 필요한가? / Does the series need differencing?

짝 슬라이드 「Can We Read This ACF the Way We Did Last Week?」 · 「Two Tests Start From Opposite Assumptions」 · 「Read the Two Tests Together」.
*Matching slides: "Can We Read This ACF the Way We Did Last Week?", "Two Tests Start From Opposite Assumptions", "Read the Two Tests Together".*

**먼저 예측하세요.** 아래 계열은 정상(stationary)으로 보이나요? 6주차처럼 ACF 에서 AR 이나 MA 의 모양을 읽을 수 있을까요?
*Predict first. Does the series below look stationary? Can you read an AR or MA shape from its ACF, as in Week 6?*

In [ ]:
r = acf(jobs, nlags=36, fft=False)[1:]                   # 시차 1 ~ 36 의 표본 ACF / sample ACF at lags 1 to 36
band = 2 / np.sqrt(len(jobs))                            # 6주차의 띠 ±2/√n / Week 6's band ±2/√n

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].plot(jobs.index.to_timestamp(), jobs.to_numpy(), color=GREEN)
ax[0].set_title("jobs in leisure and hospitality (millions)")
ax[1].bar(np.arange(1, 37), r, color=GREEN, width=0.5)
ax[1].axhline(band, color=GOLD, ls="--")
ax[1].axhline(-band, color=GOLD, ls="--")
ax[1].set_title("ACF, lags 1 to 36")
ax[1].set_xlabel("lag (months)")
plt.tight_layout()
plt.show()

print("ACF 시차 1 · 12 · 24 · 36 / ACF at lags 1, 12, 24, 36 :", np.round(r[[0, 11, 23, 35]], 2))
print(f"띠 / band : ±{band:.3f}")
# 기대값 / Expected
#   ACF 시차 1 · 12 · 24 · 36 / ACF at lags 1, 12, 24, 36 : [0.97 0.84 0.66 0.48]
#   띠 / band : ±0.133
#   덱 「Can We Read This ACF the Way We Did Last Week?」 와 같습니다. 막대가 모두 띠를 한참 넘고,
#   천천히 줄다가 12 · 24 · 36 에서 다시 솟습니다. 오르는 수준과 해마다 도는 모양이 ACF 를 덮고 있습니다.
#   Same as the deck. Every bar is far outside the band; the bars fade slowly and rise again at 12, 24 and 36.
#   The rising level and the yearly cycle cover up the ACF.

### 두 검정과 결정표 / Two tests and the decision table

덱 「Two Tests Start From Opposite Assumptions」 의 두 검정은 **출발 가정이 반대**입니다.
*The two tests on "Two Tests Start From Opposite Assumptions" start from **opposite assumptions**.*

| | ADF | KPSS |
|---|---|---|
| 출발 가정<br>Starting assumption | 충격이 전부 남는다, 즉 충격의 효과가 영영 사라지지 않는다(단위근, 5주차 계열 B 처럼)<br>The series keeps every past shock, that is, a shock's effect never fully fades (a unit root, like Series B) | 고정된 수준 둘레에서 정상이다<br>The series is stationary around a fixed level |
| 작은 p (0.05 아래)<br>Small p (below 0.05) | 출발 가정을 버린다. 옛 충격은 사라진다. 그것만으로 정상이라는 뜻은 아니다(계열 A 는 단위근이 없지만 계속 오른다)<br>Drop it. Old shocks fade. That alone does not show the series is stationary (Series A has no unit root but keeps rising) | 출발 가정을 버린다. 고정된 수준 둘레에서 정상이 아니다. 이유(직선 · 계절 · 단위근)는 그림이 보여 준다<br>Drop it. Not stationary around a fixed level; the plot shows why (a line, a yearly pattern or a unit root) |
| 큰 p<br>Large p | 버리지 못한다. 단위근일 수 있다(증명은 아님)<br>Cannot drop it. A unit root is still possible (not a proof) | 버리지 못한다. 다시 차분할 이유가 없다(증명은 아님)<br>Cannot drop it. No reason to difference again (not a proof) |

결정표(「Read the Two Tests Together」)는 두 검정의 답 네 가지 조합을 네 줄로 정리한 것입니다. 첫째 줄은 둘 다 「정상 아님」 쪽, 둘째 줄은 둘 다 「정상」 쪽입니다. 셋째 줄은 두 검정이 모두 출발 가정을 버려 서로 다른 쪽을 가리키고, 넷째 줄은 둘 다 출발 가정을 버리지 못해 검정만으로는 정할 수 없습니다(계열이 짧을 수 있습니다). TODO 1 에서 이 표를 함수로 만듭니다.
*The decision table ("Read the Two Tests Together") puts the four combinations of the two answers in four rows. Row 1: both point to "not stationary". Row 2: both point to "stationary". Row 3: both tests drop their starting assumptions, so they disagree. Row 4: neither test drops its starting assumption, so the tests alone cannot decide (the series may be too short). TODO 1 turns this table into a function.*

In [ ]:
# ---------------------------------------------------------------- TODO 1 ----
# 다음 이름을 씁니다 / Names used below
#   adf_p       : ADF 의 p 값. 출발 가정은 「충격이 전부 남는다(단위근)」
#                 ADF p-value. Starting assumption: the series keeps every shock (a unit root)
#   kpss_p      : KPSS 의 p 값. 출발 가정은 「고정된 수준 둘레에서 정상이다」
#                 KPSS p-value. Starting assumption: the series is stationary around a fixed level
#   adf_drops   : ADF 가 자기 출발 가정을 버렸나(p < 0.05) / did ADF drop its starting assumption (p below 0.05)?
#   kpss_drops  : KPSS 가 자기 출발 가정을 버렸나(p < 0.05) / did KPSS drop its starting assumption?
#   both_point_to_stationary : 두 검정이 모두 「정상」 쪽을 가리키나   <- 여기가 빈칸
#                 do both tests point to "stationary"?   <- this is the blank
#
# 참고 / Reference   and · not
#   a and b  : a 와 b 가 둘 다 참일 때만 참 / True only when both a and b are True
#   not a    : 참과 거짓을 뒤집는다 / flips True and False
#   예 / e.g.   (3 > 2) and not (1 > 2)  ->  True

def decision_row(adf_p, kpss_p):
    """Return the row (1 to 4) of the deck's decision table "Read the Two Tests Together"."""
    adf_drops = adf_p < 0.05            # 작은 p 면 「단위근」 을 버린다 / a small p drops "keeps every shock"
    kpss_drops = kpss_p < 0.05          # 작은 p 면 「정상」 을 버린다 / a small p drops "stationary"

    # TODO 1. 결정표 둘째 줄 「Both point to “stationary”」 는 adf_drops 와 kpss_drops 의 어떤 조합인가요?
    #         덱 「Two Tests Start From Opposite Assumptions」 에서 두 검정의 출발 가정을 다시 보세요. 한 줄입니다.
    # TODO 1. Which combination of adf_drops and kpss_drops is row 2, "Both point to “stationary”"?
    #         Look again at the two starting assumptions on "Two Tests Start From Opposite Assumptions". One line.
    both_point_to_stationary = None
    if both_point_to_stationary is None:
        raise NotImplementedError(BLANK_MSG)

    if both_point_to_stationary:
        return 2                        # 둘째 줄. 차분을 멈춘다 / row 2. stop differencing
    if (not adf_drops) and kpss_drops:
        return 1                        # 첫째 줄. 둘 다 「정상 아님」 쪽 / row 1. both point to "not stationary"
    if adf_drops and kpss_drops:
        return 3                        # 셋째 줄. 서로 다르다 / row 3. they disagree
    return 4                            # 넷째 줄. 둘 다 버리지 못함 / row 4. neither drops its assumption


# 한 번 불러 보기. 덱 「How Many Differences Does the Jobs Series Need?」 표 첫 줄(원계열)의 두 p 입니다.
# Call it once, with the two p-values in the first row (Original) of "How Many Differences Does the Jobs Series Need?".
print("ADF p = 0.94, KPSS p = 0.01  ->  줄 / row", decision_row(0.94, 0.01))
# 기대값 / Expected
#   ADF p = 0.94, KPSS p = 0.01  ->  줄 / row 1
#   다른 줄이 나와도 바로 아래 자가검증이 어디가 틀렸는지 알려 줍니다.
#   If you get another row, the self-check below tells you what went wrong.
# --------------------------------------------------------------------------

In [ ]:
# 자가검증 / Self-check
cases = [(0.94, 0.01, 1), (0.003, 0.10, 2), (0.01, 0.01, 3), (0.35, 0.40, 4)]
for adf_p, kpss_p, want in cases:
    got = decision_row(adf_p, kpss_p)
    assert got == want, (
        f"ADF p = {adf_p}, KPSS p = {kpss_p} 에서 정답은 {want} 번째 줄, 결과는 {got} 번째 줄입니다. "
        "ADF 의 작은 p 는 「충격이 전부 남는다」 를 버리고, KPSS 의 작은 p 는 「정상」 을 버립니다. "
        "그래서 둘 다 「정상」 쪽을 가리키려면 ADF 의 p 는 작고 KPSS 의 p 는 커야 합니다.\n"
        f"ADF p = {adf_p}, KPSS p = {kpss_p} should give row {want}, but gave row {got}. "
        "A small ADF p drops \"keeps every shock\"; a small KPSS p drops \"stationary\". "
        "So both point to \"stationary\" when the ADF p is small and the KPSS p is large.")
print("통과. 두 검정의 출발 가정이 반대라서, 같은 「작은 p」 가 서로 다른 말을 합니다.")
print("Passed. The two starting assumptions are opposite, so the same small p says different things.")

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.
# 실행한 뒤에는 바로 위의 자가검증 셀을 다시 실행하세요. 자가검증은 지금 메모리에 있는 것을 검사합니다.
# After running it, run the self-check cell just above again. The self-check tests whatever is in memory now.

def decision_row(adf_p, kpss_p):
    """Return the row (1 to 4) of the deck's decision table "Read the Two Tests Together"."""
    adf_drops = adf_p < 0.05            # 작은 p 면 「단위근」 을 버린다 / a small p drops "keeps every shock"
    kpss_drops = kpss_p < 0.05          # 작은 p 면 「정상」 을 버린다 / a small p drops "stationary"

    both_point_to_stationary = adf_drops and not kpss_drops   # 빈칸이었던 줄. ADF 는 단위근을 버리고, KPSS 는 정상을 못 버린다
                                                              # the blank. ADF drops the unit root and KPSS cannot drop "stationary"
    if both_point_to_stationary is None:
        raise NotImplementedError(BLANK_MSG)

    if both_point_to_stationary:
        return 2                        # 둘째 줄. 차분을 멈춘다 / row 2. stop differencing
    if (not adf_drops) and kpss_drops:
        return 1                        # 첫째 줄. 둘 다 「정상 아님」 쪽 / row 1. both point to "not stationary"
    if adf_drops and kpss_drops:
        return 3                        # 셋째 줄. 서로 다르다 / row 3. they disagree
    return 4                            # 넷째 줄. 둘 다 버리지 못함 / row 4. neither drops its assumption


print("ADF p = 0.94, KPSS p = 0.01  ->  줄 / row", decision_row(0.94, 0.01))
# 기대값 / Expected
#   ADF p = 0.94, KPSS p = 0.01  ->  줄 / row 1

In [ ]:
for label, reg in [("기본판 / default versions", "c"), ("추세판 / trend versions  ", "ct")]:
    a, ap, k, kp = test_pair(jobs, regression=reg)
    print(f"{label} : ADF {a:.2f} (p {fmt_p(ap)}), KPSS {k:.2f} (p {fmt_p(kp, 'kpss')})  ->  줄 / row {decision_row(ap, kp)}")
# 기대값 / Expected
#   기본판 / default versions : ADF -0.18 (p = 0.94), KPSS 3.99 (p < 0.01)  ->  줄 / row 1
#   추세판 / trend versions   : ADF -2.04 (p = 0.58), KPSS 0.54 (p < 0.01)  ->  줄 / row 1
#   기본판은 덱 「How Many Differences Does the Jobs Series Need?」 표의 첫 줄과 같은 숫자입니다.
#   The default line has the numbers of the first row of the table on "How Many Differences Does the Jobs Series Need?".

**What to notice.** 기본판은 첫째 줄입니다. 결정표 첫째 줄은 「그림이 직선을 따라 오르면 다음 장(추세판)을 먼저 보라」 고 하고, 이 계열도 오르므로 추세판도 돌렸습니다. 추세판도 첫째 줄입니다. 직선 하나를 빼는 것으로는 부족하니 차분합니다. 검정은 「아직 정상이 아니다」 까지만 말하고, 어떤 차분을 먼저 시도할지는 그림이 제안합니다. 시도한 결과는 검정 · 잔차 · 예측으로 점검합니다. 해마다 같은 모양이 돌아오니 계절 차분부터 시도합니다(2절).
*The default versions land in row 1. Row 1 says "if the plot rises along a straight line, see the next slide first", and this series rises, so we also ran the trend versions. They land in row 1 too: removing one straight line is not enough, so we difference. The tests only say "not stationary yet"; the plot suggests which difference to try first, and the tests, residuals and forecasts check the result. The same shape returns every year, so we try a seasonal difference first (section 2).*

**Try it. 5주차 계열 A 와 B.** 덱 「Can the Tests Tell Series A From Series B?」 는 기본판이 두 계열을 가르지 못하고 추세판은 가른다고 했습니다. 아래 셀로 확인하세요.
*Try it. The Week 5 series A and B. The slide "Can the Tests Tell Series A From Series B?" says the default versions cannot tell them apart and the trend versions can. Check it with the cell below.*

In [ ]:
# 5주차 실습 5절과 같은 두 계열(시드 101, 길이 100). A 는 직선 + 충격, B 는 충격이 쌓인 계열.
# The two Week 5 series (lab section 5, seed 101, length 100). A: a line plus shocks. B: accumulated shocks.
eps = np.random.default_rng(101).standard_normal(100)       # 두 계열이 같은 충격을 쓴다 / both series use the same shocks
times = np.arange(1, 101)                                   # 시각 1 ~ 100 / times 1 to 100
series_A = 0.5 * times + eps                                # 추세정상 / trend-stationary
series_B = np.cumsum(0.5 + eps)                             # 차분정상 / difference-stationary

for label, reg in [("기본판 / default", "c"), ("추세판 / trend  ", "ct")]:
    for name, x in [("A", series_A), ("B", series_B)]:
        a, ap, k, kp = test_pair(x, regression=reg)
        print(f"{label} · {name} : ADF {a:.2f} (p {fmt_p(ap)}), KPSS {k:.2f} (p {fmt_p(kp, 'kpss')})  ->  줄 / row {decision_row(ap, kp)}")
# 기대값 / Expected
#   기본판 / default · A : ADF 0.12 (p = 0.97), KPSS 2.10 (p < 0.01)  ->  줄 / row 1
#   기본판 / default · B : ADF -1.65 (p = 0.46), KPSS 2.08 (p < 0.01)  ->  줄 / row 1
#   추세판 / trend   · A : ADF -6.80 (p < 0.001), KPSS 0.06 (p > 0.10)  ->  줄 / row 2
#   추세판 / trend   · B : ADF -2.11 (p = 0.54), KPSS 0.35 (p < 0.01)  ->  줄 / row 1
#   기본판은 A 와 B 를 모두 첫째 줄(차분)로 보냅니다. A 를 차분하면 한 번이 너무 많습니다.
#   추세판에서 A 는 둘째 줄(직선 둘레에서 정상, 직선만 뺀다), B 는 그대로 첫째 줄(차분)입니다.
#   The default versions send both A and B to row 1 (difference). For A that difference is one too many.
#   With the trend versions A lands in row 2 (stationary around a line, so subtract the line) and B stays in row 1.

---
## 2. 어떤 차분을 할까? / Which differences make it stationary?

짝 슬라이드 「What If the Pattern Repeats Every 12 Months?」 · 「How Many Differences Does the Jobs Series Need?」.
*Matching slides: "What If the Pattern Repeats Every 12 Months?", "How Many Differences Does the Jobs Series Need?".*

계절 차분은 각 달에서 **작년 같은 달**을 뺍니다. 해마다 도는 월별 자료라면 계절 주기 $m = 12$ (한 주기의 관측 수, FPP-py §9.9) 이고,
*A seasonal difference subtracts **the same month one year earlier**. For monthly data with a yearly cycle the seasonal period is $m = 12$ (the number of observations in one cycle, FPP-py §9.9), and*

$$\nabla_{12}\, x_t = x_t - x_{t-12}.$$

5주차의 보통 차분 $\nabla x_t = x_t - x_{t-1}$ 은 같은 꼴에서 12 대신 1 을 쓴 것입니다.
*The ordinary difference of Week 5, $\nabla x_t = x_t - x_{t-1}$, is the same form with 1 in place of 12.*

**먼저 예측하세요.** 계절 차분 한 번이면 충분할까요, 보통 차분도 한 번 더 필요할까요?
*Predict first. Is one seasonal difference enough, or is an ordinary difference needed as well?*

In [ ]:
# ---------------------------------------------------------------- TODO 2 ----
# 다음 이름을 씁니다 / Names used below
#   x    : 월별 계열(pandas Series). 색인이 달이다 / a monthly series (pandas Series) indexed by month
#   m    : 몇 칸 앞과 비교하나. 계절 차분이면 계절 주기 12, 보통 차분이면 1
#          how many places back to compare with: the seasonal period 12, or 1 for an ordinary difference
#   sd   : 차분한 계열 x_t - x_(t-m)   <- 여기가 빈칸
#          the differenced series x_t - x_(t-m)   <- this is the blank
#
# 참고 / Reference   x.shift(k)
#   값을 모두 k 칸 뒤로 민다. 맨 앞 k 칸은 값이 없어 NaN 이 된다.
#   Moves every value k places later. The first k places have no value and become NaN.
#   예 / e.g.   pd.Series([10, 11, 13]).shift(1)  ->  [NaN, 10, 11]
#   그래서 x.shift(k) 의 t 번째 칸에는 x_(t-k) 가 있다 / so place t of x.shift(k) holds x_(t-k)

def seasonal_difference(x, m):
    """Return x_t - x_(t-m) for every month t. The first m months become NaN."""
    # TODO 2. 각 달의 값에서 m 칸 앞의 값을 빼세요. 덱 「What If the Pattern Repeats Every 12 Months?」 의 표를
    #         계열 전체에 하는 것입니다. 12 가 아니라 m 을 쓰세요(아래에서 m = 1 로도 씁니다). 한 줄입니다.
    # TODO 2. Subtract from each month the value m places back, like the table on
    #         "What If the Pattern Repeats Every 12 Months?" but for the whole series.
    #         Use m, not 12 (we also use m = 1 below). One line.
    sd = None
    if sd is None:
        raise NotImplementedError(BLANK_MSG)

    return sd
# --------------------------------------------------------------------------


m = 12                                                      # 계절 주기. 월별 자료의 연 주기 / seasonal period, a yearly cycle in monthly data
jobs_sd = seasonal_difference(jobs, m)                      # 계절 차분한 계열 / the seasonally differenced series
picked = jobs_sd.loc[["2018-01", "2018-04", "2018-07", "2018-10"]]
print("2018 - 2017, 1 · 4 · 7 · 10월 / Jan, Apr, Jul, Oct :", np.round(picked.to_numpy(), 3))
print("앞에서 빈 달 / empty months at the start :", int(jobs_sd.isna().sum()))
# 기대값 / Expected
#   2018 - 2017, 1 · 4 · 7 · 10월 / Jan, Apr, Jul, Oct : [0.313 0.245 0.278 0.304]
#   앞에서 빈 달 / empty months at the start : 12
#   덱 「What If the Pattern Repeats Every 12 Months?」 의 표와 같습니다. 작년 같은 달이 없는 처음 12개월은 비어 있습니다.
#   The same as the table on the slide. The first 12 months have no month a year earlier, so they are empty.
#   첫 줄이 [-0.422 ...] 로 시작하면 작년 같은 달이 아니라 바로 앞 달을 뺀 것입니다.
#   A first line starting [-0.422 ...] means you subtracted the previous month instead of the same month a year earlier.

In [ ]:
# 자가검증 / Self-check
assert len(jobs_sd) == len(jobs), "차분한 계열도 225개월 길이여야 합니다(앞은 NaN) / the differenced series keeps all 225 months (NaN at the start)"
assert jobs_sd.iloc[:12].isna().all() and jobs_sd.iloc[12:].notna().all(), \
    "처음 12개월만 비어 있어야 합니다 / only the first 12 months should be empty"
assert np.isclose(jobs_sd.loc["2018-01"], jobs.loc["2018-01"] - jobs.loc["2017-01"]), \
    "2018년 1월은 2018년 1월 - 2017년 1월이어야 합니다 / January 2018 must be January 2018 minus January 2017"
one = seasonal_difference(jobs, 1)
assert np.isclose(one.loc["2001-02"], jobs.loc["2001-02"] - jobs.loc["2001-01"]), \
    "m = 1 이면 바로 앞 달을 빼야 합니다. 12 를 직접 쓰지 말고 m 을 쓰세요 / with m = 1 it must subtract the previous month; use m, not 12"
print("통과. 계절 차분은 각 달에서 작년 같은 달을 뺀 것이고, m = 1 이면 보통 차분입니다.")
print("Passed. A seasonal difference subtracts the same month a year earlier; with m = 1 it is the ordinary difference.")

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.
# 실행한 뒤에는 바로 위의 자가검증 셀을 다시 실행하세요. 자가검증은 지금 메모리에 있는 것을 검사합니다.
# After running it, run the self-check cell just above again. The self-check tests whatever is in memory now.

def seasonal_difference(x, m):
    """Return x_t - x_(t-m) for every month t. The first m months become NaN."""
    sd = x - x.shift(m)         # 빈칸이었던 줄. m 칸 뒤로 민 계열을 빼면 x_t - x_(t-m)
                                # the blank. Subtracting the series moved m places later gives x_t - x_(t-m)
    if sd is None:
        raise NotImplementedError(BLANK_MSG)

    return sd


m = 12                                                      # 계절 주기. 월별 자료의 연 주기 / seasonal period, a yearly cycle in monthly data
jobs_sd = seasonal_difference(jobs, m)                      # 계절 차분한 계열 / the seasonally differenced series
picked = jobs_sd.loc[["2018-01", "2018-04", "2018-07", "2018-10"]]
print("2018 - 2017, 1 · 4 · 7 · 10월 / Jan, Apr, Jul, Oct :", np.round(picked.to_numpy(), 3))
print("앞에서 빈 달 / empty months at the start :", int(jobs_sd.isna().sum()))
# 기대값 / Expected
#   2018 - 2017, 1 · 4 · 7 · 10월 / Jan, Apr, Jul, Oct : [0.313 0.245 0.278 0.304]
#   앞에서 빈 달 / empty months at the start : 12
#   덱 「What If the Pattern Repeats Every 12 Months?」 의 표와 같습니다. 작년 같은 달이 없는 처음 12개월은 비어 있습니다.
#   The same as the table on the slide. The first 12 months have no month a year earlier, so they are empty.
#   첫 줄이 [-0.422 ...] 로 시작하면 작년 같은 달이 아니라 바로 앞 달을 뺀 것입니다.
#   A first line starting [-0.422 ...] means you subtracted the previous month instead of the same month a year earlier.

In [ ]:
a, ap, k, kp = test_pair(jobs_sd)
print(f"계절 차분 뒤 / after the seasonal difference : ADF {a:.2f} (p {fmt_p(ap)}), KPSS {k:.2f} (p {fmt_p(kp, 'kpss')})  ->  줄 / row {decision_row(ap, kp)}")

jobs_dd = seasonal_difference(jobs_sd, 1)                   # 보통 차분은 같은 함수에 m = 1 / an ordinary difference is the same function with m = 1
a, ap, k, kp = test_pair(jobs_dd)
print(f"두 차분 뒤 / after both differences          : ADF {a:.2f} (p {fmt_p(ap)}), KPSS {k:.2f} (p {fmt_p(kp, 'kpss')})  ->  줄 / row {decision_row(ap, kp)}")
print("남은 값 / values left :", int(jobs_dd.notna().sum()))

fig, ax = plt.subplots(figsize=(11, 3.0))
ax.plot(jobs_dd.index.to_timestamp(), jobs_dd.to_numpy(), color=GREEN)
ax.axhline(0, color="0.5", lw=1)
ax.set_title("after both differences (millions)")
plt.show()
# 기대값 / Expected
#   계절 차분 뒤 / after the seasonal difference : ADF -2.12 (p = 0.24), KPSS 1.15 (p < 0.01)  ->  줄 / row 1
#   두 차분 뒤 / after both differences          : ADF -3.82 (p = 0.003), KPSS 0.08 (p > 0.10)  ->  줄 / row 2
#   남은 값 / values left : 212
#   덱 「How Many Differences Does the Jobs Series Need?」 의 표와 그림과 같습니다. 계절 차분 뒤에도 첫째 줄이라
#   보통 차분을 한 번 더 했고, 두 차분 뒤 둘째 줄이라 멈춥니다. D = 1, d = 1. 두 번 차분하면 13개월이 빠져 212개가 남습니다.
#   Same as the table and plot on the slide. Still row 1 after the seasonal difference, so one ordinary difference more;
#   row 2 after both, so we stop. D = 1 and d = 1. The two differences use up 13 months, leaving 212 values.

**What to notice.** 계절 차분은 해마다 도는 모양을 지우고, 그 뒤에 남은 느린 오르내림은 보통 차분이 지웁니다. 두 차분 뒤의 값은 0 둘레의 한 띠 안에서 움직입니다.
*The seasonal difference removes the yearly shape, and the ordinary difference removes the slow drift left after it. After both, the values move around 0 within one band.*

ADF 와 KPSS 는 시차 1 의 단위근을 보는 검정이지, 계절 단위근(시차 12)의 검정이 아닙니다. 계절 차분을 먼저 시도한 근거는 그림(해마다 도는 모양이 떠돈다, 덱 「Does Every Yearly Pattern Need a Seasonal Difference?」)이고, 두 검정은 그 뒤에 보통 차분이 더 필요한지를 봅니다. 교재의 자동 절차(FPP-py §9.1 의 `nsdiffs`)는 계절 강도 측도를 따로 씁니다.
*ADF and KPSS test for a unit root at lag 1, not for a seasonal unit root at lag 12. The reason for trying the seasonal difference first is the plot (the yearly shape drifts, "Does Every Yearly Pattern Need a Seasonal Difference?"); the two tests then ask whether an ordinary difference is still needed. The textbook's automatic procedure (`nsdiffs` in FPP-py §9.1) uses a separate measure of seasonal strength.*

**Try it. 순서를 바꾸면? 한 번 더 하면?** 교재(FPP-py §9.1)는 두 차분의 순서가 결과를 바꾸지 않는다고 합니다. 그리고 덱 「What If We Difference One Time Too Many?」 는 한 번 더 차분하면 시차 1 에 −0.5 근처의 막대가 생기기 쉽고, 그 막대는 증명이 아니라 경고라고 했습니다. 둘 다 확인해 보세요.
*Try it. Swap the order, or difference once more. FPP-py §9.1 says the order of the two differences does not change the result, and "What If We Difference One Time Too Many?" says one difference too many tends to leave a bar near −0.5 at lag 1, a warning rather than proof. Check both.*

In [ ]:
other_order = seasonal_difference(seasonal_difference(jobs, 1), 12)   # 보통 차분 먼저, 계절 차분 나중 / ordinary first, seasonal second
print("두 순서의 가장 큰 차이 / largest gap between the two orders :", round(float(np.nanmax(np.abs(other_order - jobs_dd))), 12))

too_many = seasonal_difference(jobs_dd, 1)                  # 한 번 더 보통 차분 / one more ordinary difference
print("시차 1 ACF, 두 차분 뒤 / lag-1 ACF after both differences :", round(float(acf(jobs_dd.dropna(), nlags=1, fft=False)[1]), 2))
print("시차 1 ACF, 한 번 더 / lag-1 ACF after one more           :", round(float(acf(too_many.dropna(), nlags=1, fft=False)[1]), 2))
# 기대값 / Expected
#   두 순서의 가장 큰 차이 / largest gap between the two orders : 0.0
#   시차 1 ACF, 두 차분 뒤 / lag-1 ACF after both differences : 0.16
#   시차 1 ACF, 한 번 더 / lag-1 ACF after one more           : -0.52
#   순서를 바꿔도 같은 계열입니다. 한 번 더 차분하면 시차 1 이 0.16 에서 -0.52 로 떨어집니다. 필요 없는 차분이 만든 음의 상관입니다.
#   The order does not matter. One difference more drops the lag-1 ACF from 0.16 to -0.52, a negative correlation made by the extra difference.
#   덱대로 이 막대는 증명이 아니라 경고입니다. 원래부터 음의 MA(1) 인 계열도 비슷한 막대를 낼 수 있으니, 그 차분이 필요했는지(검정 · 그림)를 다시 봅니다.
#   As on the slide, this bar is a warning, not proof: a series that is a negative MA(1) to begin with can show a similar bar,
#   so go back and check whether that difference was needed (tests and plot).

---
## 3. 후보 둘을 쓰고 맞추기 / Write two candidates and fit them

짝 슬라이드 「What Memory Is Left After Both Differences?」 · 「One Name for Two Kinds of Memory」 · 「Which Candidates Are Worth Checking?」.
*Matching slides: "What Memory Is Left After Both Differences?", "One Name for Two Kinds of Memory", "Which Candidates Are Worth Checking?".*

> **전체 표본 시연 / Full-sample demonstration.** 3 · 4절은 225개월 전체로 후보를 고르고 맞춥니다. 덱의 AIC · BIC · Ljung–Box 와 같은 숫자를 찍으려는 것입니다. 시험 24개월도 들어 있으니 이 숫자로 예측 성능을 말하지 않습니다. 예측 성능은 5절이 훈련 201개월만으로 맞춘 모형으로 잽니다.
> *Sections 3 and 4 choose and fit the candidates on all 225 months, to print the same AIC, BIC and Ljung–Box numbers as the deck. The 24 test months are inside, so these numbers say nothing about forecast accuracy. Section 5 measures that, with models fitted to the 201 training months only.*

**먼저 예측하세요.** 두 번 차분한 계열의 ACF 와 PACF 에서, 어느 시차의 막대가 띠 밖으로 나올까요? 짧은 시차일까요, 12 의 배수일까요, 둘 다일까요?
*Predict first. In the ACF and PACF of the twice-differenced series, which bars will leave the band: short lags, multiples of 12, or both?*

In [ ]:
x = jobs_dd.dropna()                                        # 두 번 차분한 212개 / the 212 twice-differenced values
band = 2 / np.sqrt(len(x))                                  # 띠 ±2/√n / the band ±2/√n
r_acf = acf(x, nlags=36, fft=False)[1:]                     # 시차 1 ~ 36 / lags 1 to 36
r_pacf = pacf(x, nlags=36, method="ywm")[1:]                # 6주차와 같은 방법 / the same method as Week 6

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for a_, r_, title in [(ax[0], r_acf, "ACF"), (ax[1], r_pacf, "PACF")]:
    a_.bar(np.arange(1, 37), r_, width=0.5, color=[RED if abs(v) > band else GREEN for v in r_])
    a_.axhline(band, color=GOLD, ls="--")
    a_.axhline(-band, color=GOLD, ls="--")
    a_.set_title(f"{title} after both differences")
    a_.set_xlabel("lag (months)")
plt.tight_layout()
plt.show()

print(f"띠 / band : ±{band:.3f}")
print("ACF  띠 밖 / outside the band :", {k + 1: round(float(v), 2) for k, v in enumerate(r_acf) if abs(v) > band})
print("PACF 띠 밖 / outside the band :", {k + 1: round(float(v), 2) for k, v in enumerate(r_pacf) if abs(v) > band})
# 기대값 / Expected
#   띠 / band : ±0.137
#   ACF  띠 밖 / outside the band : {1: 0.16, 2: 0.2, 11: 0.16, 12: -0.19}
#   PACF 띠 밖 / outside the band : {1: 0.16, 2: 0.18, 12: -0.29, 24: -0.15, 36: -0.18}
#   덱 「What Memory Is Left After Both Differences?」 의 두 그림과 같습니다(빨강 = 띠 밖). 시차 11 은 띠를 겨우 넘습니다.
#   36개 막대 가운데 하나쯤은 우연히도 넘을 수 있고(6주차), 달 부분과 해 부분이 곱해지는 모형은 시차 11 · 13 에
#   작은 상관을 남기기도 합니다(덱 「What If Both Parts Are AR, or Both MA?」). 어느 쪽이든 후보를 바꿀 만큼은 아닙니다.
#   The same two plots as the slide (red = outside the band). Lag 11 is just over the band: one bar in 36 can do that by
#   chance (Week 6), and a model whose short-term and yearly parts multiply also leaves small correlations at lags 11 and 13
#   ("What If Both Parts Are AR, or Both MA?"). Either way it is not enough to change the candidates.
#   시차 12 의 -0.19 는 덱 「Why Is There Still a Spike at Lag 12?」 의 고정 모양 예가 남기는 -0.5 와 다른 모양입니다.
#   하지만 이 값 하나로 계절 차분이 꼭 맞았다고 인증되지는 않습니다. 계절 MA(1) 의 기억으로 읽고 넘어갑니다.
#   The -0.19 at lag 12 is not the -0.5 that the fixed-shape example on "Why Is There Still a Spike at Lag 12?" leaves,
#   but that one number does not certify the seasonal difference either. Read it as seasonal MA(1) memory and move on.

### 막대를 차수로 읽기 / Reading the bars as orders

- 짧은 시차(1, 2)의 막대는 달에서 달로 이어지는 기억이고, 소문자 $p, q$ 가 맡습니다. 12 의 배수의 막대는 해에서 해로 이어지는 기억이고, 대문자 $P, Q$ 가 맡습니다.
  *Bars at short lags (1, 2) are month-to-month memory, handled by the lower-case $p, q$. Bars at multiples of 12 are year-to-year memory, handled by the upper-case $P, Q$.*
- 몇 칸 뒤 끊기는 쪽이 ACF 면 MA, PACF 면 AR 로 읽습니다(6주차). 처음 고르는 후보일 뿐입니다.
  *A cut-off in the ACF suggests MA, a cut-off in the PACF suggests AR (Week 6). These are first guesses.*
- 이름은 $\mathrm{ARIMA}(p, d, q)(P, D, Q)_m$ 입니다(덱 「One Name for Two Kinds of Memory」). 우리 계열은 $d = 1$, $D = 1$, $m = 12$.
  *The name is $\mathrm{ARIMA}(p, d, q)(P, D, Q)_m$. For our series $d = 1$, $D = 1$, $m = 12$.*

**후보 A (ACF 로 읽음).** ACF 가 시차 1 · 2 에서 나오고 끊기니 MA(2), 시차 12 에서 하나니 계절 MA(1). $\mathrm{ARIMA}(0,1,2)(0,1,1)_{12}$. 교재 FPP-py §9.9 의 첫 후보와 같습니다.
*Candidate A, read from the ACF. Spikes at lags 1 and 2 then a cut-off give MA(2); one spike at 12 gives a seasonal MA(1). The same first candidate as FPP-py §9.9.*

**후보 B (PACF 로 읽음)** 는 여러분 차례입니다(TODO 3). 교재처럼 달 부분은 PACF 에서, 계절 부분은 ACF 에서 읽습니다.
*Candidate B, read from the PACF, is your turn (TODO 3). As in the textbook, read the short-term part from the PACF and the yearly part from the ACF.*

In [ ]:
# ---------------------------------------------------------------- TODO 3 ----
# 다음 이름을 씁니다 / Names used below
#   후보 하나는 (order, seasonal_order) 두 묶음이다 / a candidate is a pair of tuples (order, seasonal_order)
#   order          : (p, d, q)      달에서 달로 이어지는 부분 / the month-to-month part
#   seasonal_order : (P, D, Q, m)   해에서 해로 이어지는 부분 / the year-to-year part
#   cand_A         : ACF 로 읽은 후보(주어짐) / the candidate read from the ACF (given)
#   cand_B         : PACF 로 읽은 후보   <- 여기가 빈칸 / the candidate read from the PACF   <- this is the blank
#
# 참고 / Reference   ARIMA(p,d,q)(P,D,Q)m  ->  ((p, d, q), (P, D, Q, m))
#   예 / e.g.   ARIMA(1,1,0)(1,1,0)12  ->  ((1, 1, 0), (1, 1, 0, 12))

cand_A = ((0, 1, 2), (0, 1, 1, 12))   # MA(2): ACF 시차 1 · 2, 계절 MA(1): ACF 시차 12 / MA(2) for ACF lags 1-2, seasonal MA(1) for ACF lag 12

# TODO 3. 위 PACF 의 짧은 시차 막대(1, 2)로 달 부분을 읽고, 계절 부분은 후보 A 처럼 ACF 의 시차 12 로 읽어
#         cand_B 를 쓰세요. 차분 횟수는 2절과 같습니다. 한 줄입니다.
# TODO 3. Read the month-to-month part from the PACF spikes at short lags (1, 2) and the yearly part from the
#         ACF spike at lag 12, as in candidate A. Same differences as section 2. One line.
cand_B = None
if cand_B is None:
    raise NotImplementedError(BLANK_MSG)

print("후보 A / candidate A :", cand_A)
print("후보 B / candidate B :", cand_B)
# --------------------------------------------------------------------------

In [ ]:
# 자가검증 / Self-check
# 이 검사는 교재(FPP-py §9.9)의 읽기 하나를 정답으로 둡니다. 다르게 읽은 후보(예: AR(1), 계절 AR 추가)는 틀린 것이 아니라
# 실험 거리입니다. 아래를 통과시킨 뒤, 마지막 「Your turn」 에서 그 후보도 맞춰 견줘 보세요.
# This check takes the textbook's reading (FPP-py §9.9) as the one answer. A candidate read differently (say AR(1), or with a
# seasonal AR term) is not wrong, it is an experiment: pass the check first, then fit and compare it in "Your turn" at the end.
(p, d, q), (P, D, Q, mm) = cand_B
assert (d, D, mm) == (1, 1, 12), "차분은 d = 1, D = 1, 계절 주기 m = 12 입니다(2절) / the differences are d = 1, D = 1 and m = 12 (section 2)"
assert (p, q) == (2, 0), "PACF 는 시차 1 · 2 에서 나오고 끊깁니다. AR 차수 p 로 읽으세요 / the PACF cuts off after lags 1 and 2: read it as the AR order p"
assert (P, Q) == (0, 1), ("계절 부분은 PACF 가 아니라 ACF 의 시차 12 에서 읽습니다. 하나 → 계절 MA(1) / "
                          "read the yearly part from the ACF spike at lag 12, not the PACF: one spike -> seasonal MA(1)")
print("통과. 후보 B 는 ARIMA(2,1,0)(0,1,1)12 입니다. 교재 FPP-py §9.9 의 두 번째 후보와 같습니다.")
print("Passed. Candidate B is ARIMA(2,1,0)(0,1,1)12, the second candidate in FPP-py §9.9.")

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.
# 실행한 뒤에는 바로 위의 자가검증 셀을 다시 실행하세요. 자가검증은 지금 메모리에 있는 것을 검사합니다.
# After running it, run the self-check cell just above again. The self-check tests whatever is in memory now.

cand_A = ((0, 1, 2), (0, 1, 1, 12))   # MA(2): ACF 시차 1 · 2, 계절 MA(1): ACF 시차 12 / MA(2) for ACF lags 1-2, seasonal MA(1) for ACF lag 12

cand_B = ((2, 1, 0), (0, 1, 1, 12))   # 빈칸이었던 줄. PACF 시차 1 · 2 → AR(2), ACF 시차 12 → 계절 MA(1), d = D = 1
                                      # the blank. PACF lags 1-2 give AR(2); ACF lag 12 gives a seasonal MA(1); d = D = 1
if cand_B is None:
    raise NotImplementedError(BLANK_MSG)

print("후보 A / candidate A :", cand_A)
print("후보 B / candidate B :", cand_B)

### 맞추고 견주기 / Fit and compare

`SARIMAX` 에는 차분하지 않은 **원계열**을 넣고, 차분 횟수는 차수 안의 $d, D$ 로 알려 줍니다. 그러면 차분은 SARIMAX 가 합니다.
*Give `SARIMAX` the **original** series and tell it the differences through $d$ and $D$ in the orders; SARIMAX does the differencing itself.*

참고 / Reference   `SARIMAX(x, order=(p, d, q), seasonal_order=(P, D, Q, m)).fit(disp=False)`  ·  결과의 `.aic` · `.aicc` · `.bic` / the fit's `.aic`, `.aicc`, `.bic`

세 번째 후보 C 는 교재의 자동 탐색이 고른 $\mathrm{ARIMA}(2,1,0)(0,1,2)_{12}$ 입니다. 자동 탐색은 여러 차수를 맞춰 AICc 가 가장 낮은 것을 내놓을 뿐, 그 모형이 맞다는 증명은 아닙니다.
*The third candidate C is $\mathrm{ARIMA}(2,1,0)(0,1,2)_{12}$, the one the textbook's automatic search picked. An automatic search fits many orders and reports the lowest AICc; it proposes candidates, it does not prove a model right.*

In [ ]:
CAND_C = ((2, 1, 0), (0, 1, 2, 12))                         # 교재(FPP-py §9.9) 자동 탐색의 후보 / the textbook's automatic-search candidate
candidates = {"A": cand_A, "B": cand_B, "C": CAND_C}


def label(cand):
    """((0, 1, 2), (0, 1, 1, 12)) -> 'ARIMA(0,1,2)(0,1,1)12', the deck's way of writing it."""
    (p, d, q), (P, D, Q, mm) = cand
    return f"ARIMA({p},{d},{q})({P},{D},{Q}){mm}"


def fit_sarima(x, cand):
    """후보 하나를 계열 x 에 맞춘다 / fit one candidate to the series x."""
    order, seasonal_order = cand
    return SARIMAX(np.asarray(x, dtype=float), order=order, seasonal_order=seasonal_order).fit(disp=False)


fits_full = {name: fit_sarima(jobs, cand) for name, cand in candidates.items()}   # 225개월 전체 / all 225 months
print("후보 / candidate                AIC     AICc     BIC")
for name, f in fits_full.items():
    print(f"{name}  {label(candidates[name]):24s} {f.aic:7.1f}  {f.aicc:7.1f}  {f.bic:7.1f}")
# 기대값 / Expected
#   후보 / candidate                AIC     AICc     BIC
#   A  ARIMA(0,1,2)(0,1,1)12     -774.9   -774.7   -761.4
#   B  ARIMA(2,1,0)(0,1,1)12     -776.2   -776.0   -762.7
#   C  ARIMA(2,1,0)(0,1,2)12     -779.1   -778.8   -762.3
#   덱 「Which Candidates Are Worth Checking?」 의 AIC · BIC 와 같습니다. AIC 는 C 가, BIC 는 B 가 가장 낮습니다.
#   교재 표의 AICc(-774 · -776 · -779)와는 1 안쪽으로 다릅니다. 교재는 다른 패키지(statsforecast)로 맞췄고, 순서는 같습니다.
#   같은 자료 · 같은 차분의 후보끼리만 이렇게 견줄 수 있습니다.
#   The AIC and BIC on the slide. AIC prefers C and BIC prefers B. The textbook's AICc values (-774, -776, -779) differ
#   by less than 1 because it fits with another package (statsforecast); the ranking is the same.
#   Candidates can be compared like this only on the same data with the same differences.

---
## 4. 잔차에 무늬가 남았나? / Did the model leave a pattern behind?

짝 슬라이드 「Did the Model Leave a Pattern Behind?」.
*Matching slide: "Did the Model Leave a Pattern Behind?".*

> **전체 표본 시연 / Full-sample demonstration.** 3절과 같은, 225개월 전체로 맞춘 모형의 잔차입니다.
> *These are the residuals of the same all-225-month fits as section 3.*

잔차 = 관측값 − 한 걸음 앞 예측입니다. 모형이 기억을 다 썼다면 잔차는 백색잡음처럼 보입니다. Ljung–Box 검정(FPP-py §5.4)은 잔차의 시차 여러 개를 한꺼번에 봅니다. 여기서는 24 = 2 × 12 개입니다.
*Residual = observed value − one-step prediction. If a model used all the memory, its residuals look like white noise. The Ljung–Box test (FPP-py §5.4) checks many residual lags together, here 24 = 2 × 12.*

- 출발 가정은 「남은 자기상관이 없다」 입니다. **여기서는 큰 p 가 좋은 소식**입니다(통과). 작은 p 면 무늬가 남은 것이니 3절로 돌아갑니다.
  *The starting assumption is "no autocorrelation is left". **Here a large p is good news** (the model passes). A small p means a pattern is left: go back to section 3.*
- 처음 13개 잔차는 뺍니다. 차분 13칸($d + D \times m = 1 + 1 \times 12$)만큼은 앞선 값이 없어 잔차가 시작값일 뿐이기 때문입니다.
  *Drop the first 13 residuals: for the 13 months used up by the differences ($d + D \times m = 1 + 1 \times 12$) there is no earlier data, so those residuals are only start-up values.*
- `model_df` 는 모형이 추정한 ARMA 계수의 수 $p + q + P + Q$ 입니다. 검정이 그만큼을 자유도에서 뺍니다.
  *`model_df` is the number of ARMA coefficients the model estimated, $p + q + P + Q$; the test takes that many off its degrees of freedom.*

**먼저 예측하세요.** 세 후보 가운데 이 검정에서 떨어지는 것이 있을까요?
*Predict first. Will any of the three candidates fail this check?*

In [ ]:
res_C = fits_full["C"].resid[13:]                           # 후보 C 의 잔차. 처음 13개(d + D·m = 1 + 12)는 시작값이라 뺀다 / candidate C's residuals without the 13 (d + D·m = 1 + 12) start-up values
r = acf(res_C, nlags=36, fft=False)[1:]
band = 2 / np.sqrt(len(res_C))

fig, ax = plt.subplots(figsize=(11, 3.0))
ax.bar(np.arange(1, 37), r, width=0.5, color=[RED if abs(v) > band else GREEN for v in r])
ax.axhline(band, color=GOLD, ls="--")
ax.axhline(-band, color=GOLD, ls="--")
ax.set_title("residual ACF of candidate C, lags 1 to 36")
ax.set_xlabel("lag (months)")
plt.show()
print("C 잔차 ACF 띠 밖 / outside the band :", {k + 1: round(float(v), 2) for k, v in enumerate(r) if abs(v) > band})

for name, cand in candidates.items():
    (p, d, q), (P, D, Q, mm) = cand
    lb = acorr_ljungbox(fits_full[name].resid[13:], lags=[24], model_df=p + q + P + Q)
    print(f"{name} : Ljung-Box p = {lb['lb_pvalue'].iloc[0]:.2f}   (자유도 / degrees of freedom {24 - (p + q + P + Q)})")
# 기대값 / Expected
#   C 잔차 ACF 띠 밖 / outside the band : {11: 0.15}
#   A : Ljung-Box p = 0.53   (자유도 / degrees of freedom 21)
#   B : Ljung-Box p = 0.72   (자유도 / degrees of freedom 21)
#   C : Ljung-Box p = 0.77   (자유도 / degrees of freedom 20)
#   덱 「Did the Model Leave a Pattern Behind?」 와 같습니다. 셋 다 큰 p 라 뚜렷한 무늬가 남지 않았습니다(통과).
#   시차 11 의 작은 막대는 교재도 같은 자리에서 보고합니다. 큰 p 는 「잔차가 독립이다」 의 증명이 아닙니다.
#   Same as the slide. All three p-values are large: no clear pattern is left. The small bar at lag 11 is the one
#   FPP-py §9.9 reports too. A large p is not a proof that the residuals are independent.

---
## 5. 마지막 24개월에서 어느 예측이 이기나? / Which forecast wins on the last 24 months?

짝 슬라이드 「What Should an Error Measure Tell Us?」 · 「How Do We Test Without Using the Future?」 · 「Did SARIMA Beat the Seasonal Naive Forecast?」.
*Matching slides: "What Should an Error Measure Tell Us?", "How Do We Test Without Using the Future?", "Did SARIMA Beat the Seasonal Naive Forecast?".*

> **훈련 201개월 · 시험 24개월 / 201 training months, 24 test months.** 여기부터 시험 달은 맞추는 데 쓰지 않습니다. 후보의 차수는 3절에서 225개월로 정했으므로, 아래 둘째 줄과 Try it 이 그 점을 따로 확인합니다.
> *From here on, the test months are never used for fitting. The candidates' orders were set in section 3 from all 225 months, so the second bullet below and the Try it check that point separately.*

- 마지막 24개월(2017-10 ~ 2019-09)을 시험 구간으로 떼어 두고, 그 앞 201개월로만 맞춥니다. 시험 구간을 보고 모형을 고치면 더 이상 시험이 아닙니다.
  *Hold out the last 24 months (2017-10 to 2019-09) as the test and fit on the 201 months before them only. Changing the model after seeing the test months would spoil the test.*
- **후보를 고르는 것도 훈련 달만으로** 해야 합니다. 우리는 A · B · C 를 225개월 전체로 골랐지만, 훈련 달만으로 그림을 다시 읽어도 같은 A · B 가 나오고(아래 Try it, 그리고 시작점마다 다시 고르는 선택 D), C 를 고른 교재의 자동 탐색도 훈련 달만으로 돌리면 같은 C 를 고릅니다(교수자 쪽에서 statsforecast 로 확인. Colab 에 기본으로 없어 여기서는 돌리지 않습니다).
  *Choosing the candidates must also use the training months only. We chose A, B and C from all 225 months, but re-reading the plots from the training months gives the same A and B (Try it below, and Optional D, which chooses again at every starting point), and the textbook's automatic search that picked C picks the same C from the training months (checked on the instructor side with statsforecast; it is not preinstalled in Colab, so it is not run here).*
- 비교 기준 = 계절 나이브 예측. 각 달을 작년 같은 달 값으로 예측합니다. 2017년 9월에 24개월을 한꺼번에 예측하므로 훈련의 마지막 12개월이 두 번 쓰입니다.
  *The yardstick is the seasonal naive forecast: each month is predicted by the same month one year earlier. All 24 months are forecast at once from September 2017, so the last 12 training months are used twice.*
- MAE · RMSE 는 자료와 같은 단위(백만 명)이고, MASE 는 MAE 를 기준 오차로 나눈 단위 없는 값입니다(FPP-py §5.8).
  *MAE and RMSE are in the data's units (millions); MASE divides the MAE by a yardstick error and has no units (FPP-py §5.8).*

**먼저 예측하세요.** SARIMA 세 후보가 계절 나이브를 이길까요? 계절 나이브 자신의 MASE 는 1 일까요?
*Predict first. Will the three SARIMA candidates beat the seasonal naive forecast? Is the seasonal naive forecast's own MASE equal to 1?*

In [ ]:
TRAIN_END = "2017-09"                                       # 훈련의 마지막 달. 바꾸면 시험 길이가 바뀐다(마지막 Your turn) / last training month; change it to change the test length
train = jobs.loc[:TRAIN_END]                                # 훈련 201개월 / 201 training months
test = jobs.loc[TRAIN_END:].iloc[1:]                        # 그 뒤 전부, 시험 24개월 / everything after it, the 24 test months
H = len(test)                                               # 시험 길이 / the test length

# 참고 / Reference   np.tile(a, k) : a 를 k 번 이어 붙인다 / repeats a k times.   예 / e.g. np.tile([1, 2], 2) -> [1, 2, 1, 2]
sn_forecast = np.tile(train.to_numpy()[-12:], int(np.ceil(H / 12)))[:H]   # 작년 같은 달. 마지막 12개월을 시험 길이만큼 되풀이(24개월이면 두 번) / last year's same month: the last 12 months repeated to the test length (twice for 24)
test_sn_errors = test.to_numpy() - sn_forecast              # 시험 구간의 계절 나이브 오차 / seasonal naive errors on the test months

y_tr = train.to_numpy()
train_sn_errors = y_tr[12:] - y_tr[:-12]                    # 훈련 구간의 한 걸음 계절 나이브 오차 y_t - y_(t-12) / one-step seasonal naive errors on the training months

print("훈련 / training :", len(train), " 시험 / test :", len(test))
print("훈련 구간 한 걸음 오차 / one-step training errors :", len(train_sn_errors))
print("시험 처음 세 달의 계절 나이브 예측 / seasonal naive, first three test months :", np.round(sn_forecast[:3], 3))
# 기대값 / Expected
#   훈련 / training : 201  시험 / test : 24
#   훈련 구간 한 걸음 오차 / one-step training errors : 189
#   시험 처음 세 달의 계절 나이브 예측 / seasonal naive, first three test months : [15.721 15.55  15.489]
#   훈련 오차는 작년 같은 달이 있는 2002-01 부터라 201 - 12 = 189 개입니다.
#   The training errors start in January 2002, the first month with a month one year earlier: 201 - 12 = 189 of them.

In [ ]:
# ---------------------------------------------------------------- TODO 4 ----
# 다음 이름을 씁니다 / Names used below
#   train_sn_errors : 훈련 구간에서 「작년 같은 달」 한 걸음 예측의 오차 189개
#                     the 189 one-step seasonal naive errors on the training months
#   test_sn_errors  : 시험 24개월에서 계절 나이브 예측의 오차 / the seasonal naive errors on the 24 test months
#   mae(e)          : 오차의 절댓값 평균(준비 셀) / mean absolute error (setup cell)
#   scale           : MASE 의 분모, 기준 예측의 MAE   <- 여기가 빈칸
#                     MASE's denominator, the yardstick's MAE   <- this is the blank
#
# 참고 / Reference   MASE = (예측의 MAE) / scale   ·   예 / e.g.  mae([0.1, -0.3, 0.2])  ->  0.2

# TODO 4. MASE 의 분모는 어느 오차의 MAE 인가요? 덱 「What Should an Error Measure Tell Us?」 의 MASE 카드를 보고
#         그 MAE 를 scale 에 넣으세요. 한 줄입니다.
# TODO 4. Which errors give MASE's denominator? Read the MASE card on "What Should an Error Measure Tell Us?"
#         and put that MAE into scale. One line.
scale = None
if scale is None:
    raise NotImplementedError(BLANK_MSG)

print(f"MASE 분모 / MASE denominator : {scale:.3f}")
print(f"계절 나이브의 MASE / seasonal naive MASE : {mae(test_sn_errors) / scale:.2f}")
# 기대값 / Expected
#   MASE 분모 / MASE denominator : 0.323
#   계절 나이브의 MASE / seasonal naive MASE : 1.51
#   덱과 같습니다. 분모는 훈련 구간의 한 걸음 오차라, 24걸음까지 내다본 시험에서는 계절 나이브 자신도 1 이 아닙니다.
#   Same as the deck. The yardstick is one-step errors on the training months, so on a test that looks up to 24 months
#   ahead even the seasonal naive forecast does not score 1.
#   분모가 0.488, MASE 가 1.00 으로 나오면 시험 구간의 오차로 나눈 것입니다.
#   A denominator of 0.488 and a MASE of 1.00 mean you divided by the test-month errors.
# --------------------------------------------------------------------------

In [ ]:
# 자가검증 / Self-check
assert abs(scale - mae(test_sn_errors)) > 0.01, ("시험 구간의 오차로 나누면 계절 나이브의 MASE 가 늘 1 이 됩니다. 훈련 구간의 한 걸음 오차를 쓰세요 / "
                                                 "dividing by the test-month errors always gives the seasonal naive a MASE of 1; use the one-step training errors")
if TRAIN_END == "2017-09":                                  # 기본 분할(시험 24개월)일 때만 숫자를 맞춰 본다 / the number is checked only for the default split (24 test months)
    assert abs(scale - 0.323) < 0.0005, "분모는 0.323 근처여야 합니다(덱 「What Should an Error Measure Tell Us?」) / the denominator should be about 0.323"
else:
    print(f"훈련 끝을 {TRAIN_END} 로 바꿨으므로 0.323 대신 지금 분모 {scale:.3f} 를 봅니다 / the training end is {TRAIN_END}, so the denominator is {scale:.3f} instead of 0.323")
print("통과. MASE 의 분모는 훈련 구간에서 계절 나이브가 한 걸음 앞을 맞힌 오차의 MAE 입니다.")
print("Passed. MASE divides by the MAE of the seasonal naive forecast one step ahead on the training months.")

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.
# 실행한 뒤에는 바로 위의 자가검증 셀을 다시 실행하세요. 자가검증은 지금 메모리에 있는 것을 검사합니다.
# After running it, run the self-check cell just above again. The self-check tests whatever is in memory now.

scale = mae(train_sn_errors)          # 빈칸이었던 줄. 기준은 훈련 구간의 한 걸음 계절 나이브 MAE (FPP-py §5.8)
                                      # the blank. The yardstick is the one-step seasonal naive MAE on the training months
if scale is None:
    raise NotImplementedError(BLANK_MSG)

print(f"MASE 분모 / MASE denominator : {scale:.3f}")
print(f"계절 나이브의 MASE / seasonal naive MASE : {mae(test_sn_errors) / scale:.2f}")
# 기대값 / Expected
#   MASE 분모 / MASE denominator : 0.323
#   계절 나이브의 MASE / seasonal naive MASE : 1.51
#   덱과 같습니다. 분모는 훈련 구간의 한 걸음 오차라, 24걸음까지 내다본 시험에서는 계절 나이브 자신도 1 이 아닙니다.
#   Same as the deck. The yardstick is one-step errors on the training months, so on a test that looks up to 24 months
#   ahead even the seasonal naive forecast does not score 1.
#   분모가 0.488, MASE 가 1.00 으로 나오면 시험 구간의 오차로 나눈 것입니다.
#   A denominator of 0.488 and a MASE of 1.00 mean you divided by the test-month errors.

In [ ]:
errors = {"Seasonal naive": test_sn_errors}
converged = {"Seasonal naive": "-"}                         # 적합이 없으니 해당 없음 / nothing to fit
for name, cand in candidates.items():
    fit = fit_sarima(train, cand)                           # 훈련 201개월로만 맞춘다 / fit on the 201 training months only
    errors[f"{name}  {label(cand)}"] = test.to_numpy() - fit.forecast(H)   # 시험 달 전부를 한꺼번에 / all the test months at once
    converged[f"{name}  {label(cand)}"] = "yes" if fit.mle_retvals.get("converged", True) else "no"   # 최적화가 끝까지 갔나 / did the optimizer finish?

print("예측 / forecast                     MAE    RMSE   MASE   수렴 / converged")
for name, e in errors.items():
    print(f"{name:32s} {mae(e):.3f}  {rmse(e):.3f}  {mae(e) / scale:.2f}   {converged[name]}")

# 마지막 열에 「no」 가 있으면 그 적합만 한 번 더 점검합니다(준비 셀의 recheck). 다시 맞춘 로그가능도가 0.5 넘게 좋아지지 않고
# 예측이 0.01백만 안에서 움직이면 표를 그대로 읽습니다. 그렇지 않으면 그 후보의 숫자는 보류합니다. 두 문턱은 경험 규칙입니다.
# If the last column shows "no", that fit is checked once more (recheck from the setup cell). If the refit improves the
# log-likelihood by less than 0.5 and moves the forecast by less than 0.01 million, read the table as it is; otherwise hold
# that candidate's numbers. The two thresholds are rules of thumb.
for name, cand in candidates.items():
    key = f"{name}  {label(cand)}"
    if converged[key] == "no":
        llf1, llf2, gap = recheck(train, cand, H)
        verdict = ("표를 그대로 읽어도 됩니다 / the table stands" if llf2 - llf1 < 0.5 and gap < 0.01
                   else "적합 상태 점검 필요: 다른 길의 적합이 더 좋거나 예측이 다릅니다 / fit needs attention: the other route fits better or forecasts differently")
        print(f"{key}: 로그가능도 / log-likelihood {llf1:.2f} -> {llf2:.2f}, 예측 최대 차이 / largest forecast gap {gap:.4f}  ->  {verdict}")
# 기대값 / Expected
#   예측 / forecast                     MAE    RMSE   MASE   수렴 / converged
#   Seasonal naive                   0.488  0.523  1.51   -
#   A  ARIMA(0,1,2)(0,1,1)12         0.086  0.096  0.27   yes
#   B  ARIMA(2,1,0)(0,1,1)12         0.144  0.152  0.45   yes
#   C  ARIMA(2,1,0)(0,1,2)12         0.170  0.180  0.53   yes
#   덱 「Did SARIMA Beat the Seasonal Naive Forecast?」 의 왼쪽 표와 같습니다. 세 후보 모두 계절 나이브를 크게 이깁니다.
#   보류 한 번에서는 A, B, C 순서로 AIC 의 순서와 반대지만, 하나의 시험 구간이라 이 순서에 큰 뜻을 두지 마세요(선택 B).
#   The left table on the slide. All three candidates beat the seasonal naive forecast by a wide margin. On this one
#   holdout the order is A, B, C, the reverse of AIC, but one test period says little about the order (Optional B).
#   셋째 자리는 환경(OS · 수치 라이브러리 · statsmodels 판)에 따라 조금 다를 수 있습니다. 이 노트북을 만든 Linux 의 statsmodels 0.15.0 은
#   B 0.144 인데, 같은 0.15.0 이라도 Windows 에서는 B 0.145 가 보고되었고, 0.14.4 에서는 B 0.145 · C 0.172 · 0.181 이었습니다(직접 확인).
#   적합이 멈추는 자리가 환경마다 아주 조금 달라서이고, 결론은 같습니다. 두 Linux 판을 견주면 이 노트북의 다른 숫자는 모두 같습니다.
#   The third decimal can differ a little with the environment (OS, numerical libraries, statsmodels version). The Linux
#   statsmodels 0.15.0 that built this notebook gives B 0.144; the same 0.15.0 on Windows was reported as B 0.145, and
#   0.14.4 gave B 0.145 and C 0.172, 0.181 (checked). Each environment stops the fit at a very slightly different point;
#   the conclusions are the same, and between the two Linux versions every other number in this notebook is identical.
#   마지막 열이 「no」 이거나 ConvergenceWarning 이 보이면: 최적화가 정해진 허용 오차 안에서 멈추지 못했다는 뜻입니다.
#   어느 적합이 그러는지도 환경에 따라 다릅니다(이 노트북을 만든 Linux 0.15.0 에서는 이 표의 셋 모두 수렴하고 선택 B 의 적합 하나만
#   「no」, Windows 0.15.0 에서는 여기의 C 가 「no」 였다는 보고가 있음). 그럴 때 위 점검 줄이 다른 길로 다시 맞춰 견줍니다.
#   지금까지 본 경우는 모두 「표를 그대로 읽어도 됩니다」 였습니다. 경고를 숨기지 말고 점검 줄을 읽고 지나가세요.
#   A "no" in the last column, or a ConvergenceWarning, means the optimizer did not stop within its set tolerance. Which fit
#   does that also depends on the environment (on the Linux 0.15.0 that built this notebook all three fits here converge and
#   only one fit in Optional B says "no"; on Windows 0.15.0, C here was reported as "no"). The check line above then refits
#   by another route and compares. Every case seen so far came out as "the table stands": read the check line, do not
#   silence the warning, and move on.

**What to notice.** 모형을 쓸 값어치는 분명합니다(계절 나이브 오차의 1/3 ~ 1/6). 후보 셋 사이의 차이는 시험 설계에 따라 흔들릴 만큼 작습니다. 그래서 한 번의 보류 결과로 고르지 않습니다.
*The model clearly earns its place (a third to a sixth of the seasonal naive error). The differences among the three candidates are small enough to change with the test design, so we do not pick one from a single holdout.*

**Try it. 훈련 달 확인 / Training-only check.** 3절의 그림을 훈련 201개월만으로 다시 그려, 같은 후보를 읽게 되는지 보세요. 교재가 쓴 자동 탐색(statsforecast)도 훈련 달만으로 돌리면 같은 C 를 고릅니다. Colab 에 기본으로 없는 패키지라 여기서는 돌리지 않습니다.
*Try it. Training-only check. Redraw section 3's plots from the 201 training months and see whether you read the same candidates. The textbook's automatic search (statsforecast) also picks the same C from the training months; it is not preinstalled in Colab, so we do not run it here.*

In [ ]:
x_tr = seasonal_difference(seasonal_difference(train, 12), 1).dropna()   # 훈련 달만 두 번 차분 / the training months, differenced twice
band_tr = 2 / np.sqrt(len(x_tr))
r1 = acf(x_tr, nlags=36, fft=False)[1:]
r2 = pacf(x_tr, nlags=36, method="ywm")[1:]
print(f"띠 / band : ±{band_tr:.3f}")
print("ACF  띠 밖 / outside the band :", {k + 1: round(float(v), 2) for k, v in enumerate(r1) if abs(v) > band_tr})
print("PACF 띠 밖 / outside the band :", {k + 1: round(float(v), 2) for k, v in enumerate(r2) if abs(v) > band_tr})
# 기대값 / Expected
#   띠 / band : ±0.146
#   ACF  띠 밖 / outside the band : {1: 0.23, 2: 0.2, 11: 0.19, 12: -0.17, 22: -0.15, 23: -0.16, 36: -0.15}
#   PACF 띠 밖 / outside the band : {1: 0.23, 2: 0.16, 12: -0.29, 36: -0.15}
#   ACF 와 PACF 모두 짧은 시차 1 · 2 와 12 의 막대가 그대로 띠 밖이라 같은 A 와 B 를 읽습니다.
#   시차 11 은 전체 자료(3절)에서처럼 또 띠 밖이고, 22 · 23 · 36 은 띠를 겨우 넘습니다. 3절에서처럼 이런 막대만으로는
#   후보를 바꾸지 않습니다. 36개 가운데 몇 개쯤은 우연히도 넘을 수 있고, 남은 무늬는 4절의 잔차 점검이 봅니다.
#   In both plots the bars at the short lags 1, 2 and at 12 are still outside the band, so we read the same A and B.
#   Lag 11 is outside again, as with all the data (section 3), and 22, 23 and 36 are just over the band. As in section 3,
#   such bars alone do not change the candidates: a few in 36 can cross by chance, and section 4's residual check
#   looks at whatever pattern is left.

---
## 6. 선택 / Optional

수업 시간에 하지 않아도 됩니다. 접힌 셀을 펴서 실행하세요.
*Not needed in class. Unfold a cell and run it.*

- **A. 주가** 덱 「What Do the Tests Say About a Stock Price?」. 교재(FPP-py §9.1)의 구글 2015년 종가 252일.
  *A. A stock price. Google's 252 closing prices in 2015, the FPP-py §9.1 example.*
- **B. Rolling origin** 덱 「Did SARIMA Beat the Seasonal Naive Forecast?」 오른쪽 표. 시작점 7개 × 12개월(약 20~60초).
  *B. A rolling origin, the right-hand table on the same slide: 7 starting points × 12 months (about 20 to 60 seconds).*
- **C. 3년 앞** 덱 「Three Years of Forecasts for the Jobs Series」. 225개월 전체로 C 를 맞춰 36개월을 예측하고, 구간 폭과 2020년에 실제로 일어난 일을 봅니다.
  *C. Three years ahead. Fit C to all 225 months, forecast 36 months, and look at the interval widths and at what actually happened in 2020.*
- **D. 시작점마다 후보 다시 고르기** 덱 「Did SARIMA Beat the Seasonal Naive Forecast?」 의 「Choosing again before each test gives the same A, B and C」. B 의 시작점 7개마다 1 · 2 · 3절을 다시 돌립니다.
  *D. Choosing the candidates again at every starting point: the slide's "Choosing again before each test gives the same A, B and C". Sections 1 to 3 are rerun at each of B's seven starting points.*

In [ ]:
#@title ▶ Optional A. 주가: 가격과 하루 변화 / A stock price: the price and its daily change { display-mode: "form" }
# 구글(GOOG) 2015년 일별 종가, 미국 달러. 교재와 같은 자료(tsibbledata 패키지의 gafa_stock).
# Google (GOOG) daily closing prices in 2015, US dollars. The textbook's data (gafa_stock in the tsibbledata package).
GOOG_2015 = [
    521.937744, 511.057617, 499.212799, 498.357513, 499.928864, 493.454498,
    489.854309, 493.464447, 498.128784, 499.043732, 505.299316, 504.125763,
    515.204773, 531.465332, 536.994873, 532.280823, 515.791565, 507.208801,
    507.865204, 531.594604, 525.587646, 526.343506, 519.898987, 524.692566,
    528.093872, 524.941223, 534.001343, 533.036682, 539.958557, 546.00531,
    539.86908, 536.746277, 539.901306, 536.000366, 528.998901, 533.156006,
    540.893433, 552.43988, 555.343933, 568.213074, 570.500488, 570.231995,
    572.181274, 564.578125, 565.736694, 551.972473, 548.163452, 552.469727,
    544.324585, 551.47522, 547.825317, 556.437866, 554.936157, 557.293213,
    555.751648, 567.069397, 555.726807, 552.131592, 545.338989, 549.008789,
    545.000854, 539.590576, 532.59906, 533.827332, 534.080933, 538.645813,
    537.820374, 537.054565, 536.219177, 527.487183, 529.617859, 530.87854,
    521.181885, 532.44989, 531.049988, 536.413086, 544.006287, 561.967468,
    555.369995, 553.679993, 549.080017, 537.340027, 537.900024, 540.780029,
    530.799988, 524.219971, 530.700012, 538.219971, 535.700012, 529.039978,
    529.619995, 538.400024, 533.849976, 532.299988, 537.359985, 539.27002,
    542.51001, 540.109985, 532.320007, 539.789978, 539.780029, 532.109985,
    533.98999, 539.179993, 540.309998, 536.700012, 533.330017, 526.830017,
    526.690002, 536.690002, 534.609985, 532.330017, 527.200012, 528.150024,
    529.26001, 536.72998, 536.690002, 538.190002, 540.47998, 537.840027,
    535.22998, 531.690002, 521.52002, 520.51001, 521.840027, 523.400024,
    522.859985, 525.02002, 516.830017, 520.679993, 530.130005, 546.549988,
    561.099976, 560.219971, 579.849976, 672.929993, 663.02002, 662.299988,
    662.099976, 644.280029, 623.559998, 627.26001, 628, 631.929993,
    632.590027, 625.609985, 631.210022, 629.25, 643.780029, 642.679993,
    635.299988, 633.72998, 660.780029, 659.559998, 656.450012, 657.119995,
    660.869995, 656.130005, 660.900024, 646.830017, 612.47998, 589.609985,
    582.059998, 628.619995, 637.609985, 630.380005, 618.25, 597.789978,
    614.340027, 606.25, 600.700012, 614.659973, 612.719971, 621.349976,
    625.77002, 623.23999, 635.140015, 635.97998, 642.900024, 629.25,
    635.440002, 622.690002, 622.359985, 625.799988, 611.969971, 594.890015,
    594.969971, 608.419983, 611.289978, 626.909973, 641.469971, 645.440002,
    642.359985, 639.159973, 643.609985, 646.669983, 652.299988, 651.159973,
    661.73999, 662.200012, 666.099976, 650.280029, 642.609985, 651.789978,
    702, 712.780029, 708.48999, 712.950012, 716.919983, 710.809998,
    721.109985, 722.159973, 728.109985, 731.25, 733.76001, 724.890015,
    728.320007, 735.400024, 731.22998, 717, 728.960022, 725.299988,
    740, 738.409973, 756.599976, 755.97998, 748.280029, 748.150024,
    750.26001, 742.599976, 767.039978, 762.380005, 752.539978, 766.809998,
    763.25, 762.369995, 751.609985, 749.460022, 738.869995, 747.77002,
    743.400024, 758.090027, 749.429993, 739.309998, 747.77002, 750,
    750.309998, 748.400024, 762.51001, 776.599976, 771, 758.880005,
]
price = np.array(GOOG_2015)
change = np.diff(price)                                     # 하루 변화 ∇x_t / the daily change
for name, x in [("가격 / price        ", price), ("하루 변화 / change  ", change)]:
    a, ap, k, kp = test_pair(x, kpss_lags=5)                # 교재처럼 KPSS 시차 5 / KPSS lags 5, as in the textbook
    print(f"{name}: ADF {a:.2f} (p {fmt_p(ap)}), KPSS {k:.2f} (p {fmt_p(kp, 'kpss')})  ->  줄 / row {decision_row(ap, kp)}")
# 기대값 / Expected
#   가격 / price        : ADF -0.44 (p = 0.90), KPSS 3.56 (p < 0.01)  ->  줄 / row 1
#   하루 변화 / change  : ADF -14.27 (p < 0.001), KPSS 0.10 (p > 0.10)  ->  줄 / row 2
#   덱의 표와 같습니다. 교재 FPP-py §9.1 이 같은 설정으로 찍은 KPSS 통계량도 3.561(가격)과 0.099(하루 변화)입니다.
#   가격은 첫째 줄, 하루 변화는 둘째 줄이라 한 번 차분으로 멈춥니다(d = 1).
#   The deck's table. FPP-py §9.1 prints the same KPSS statistics with this setting: 3.561 (price) and 0.099 (daily change).
#   The price is row 1 and the daily change row 2, so one difference is enough (d = 1).

In [ ]:
#@title ▶ Optional B. Rolling origin, 시작점 7개 × 12개월 / 7 starting points × 12 months { display-mode: "form" }
# 2015-09 부터 6개월마다 시작점을 옮기며, 그때까지의 자료로만 다시 맞춰 다음 12개월을 예측한다.
# Move the starting point every 6 months from 2015-09; each time refit on the data up to that point and forecast 12 months.
from statsmodels.tools.sm_exceptions import ConvergenceWarning
starts = ["2015-09", "2016-03", "2016-09", "2017-03", "2017-09", "2018-03", "2018-09"]
scores = {"Seasonal naive": [], "A": [], "B": [], "C": []}
not_converged = []                                          # 끝까지 가지 못한 적합의 (후보, 시작점) / (candidate, start) of fits that did not finish
with warnings.catch_warnings():
    # 수렴 경고만 끄고 아래에서 세어서 보여 줍니다. 다른 경고는 그대로 보입니다.
    # Only the convergence warning is silenced; it is counted and shown below. Any other warning still shows.
    warnings.simplefilter("ignore", ConvergenceWarning)
    for s in starts:
        past, future = jobs.loc[:s], jobs.loc[s:].iloc[1:13]   # s 까지 / up to s,  그 뒤 12개월 / the next 12 months
        scores["Seasonal naive"].append(mae(future.to_numpy() - past.to_numpy()[-12:]))
        for name, cand in candidates.items():
            fit = fit_sarima(past, cand)
            scores[name].append(mae(future.to_numpy() - fit.forecast(12)))
            if not fit.mle_retvals.get("converged", True):
                not_converged.append((name, s))
for name, v in scores.items():
    print(f"{name:15s} 평균 MAE / mean MAE : {np.mean(v):.3f}")
print("21번의 적합 가운데 끝까지 가지 못한 것 / fits out of 21 that did not converge :", not_converged if not_converged else "없음 / none")
for name, s in not_converged:                               # 끝까지 못 간 적합마다 다른 길로 다시 맞춰 견준다(5절과 같은 점검) / refit each unfinished fit by another route and compare (the section 5 check)
    llf1, llf2, gap = recheck(jobs.loc[:s], candidates[name], 12)
    verdict = "표를 그대로 읽어도 됩니다 / the table stands" if llf2 - llf1 < 0.5 and gap < 0.01 else "적합 상태 점검 필요 / fit needs attention"
    print(f"   {name} @ {s}: 로그가능도 / log-likelihood {llf1:.2f} -> {llf2:.2f}, 예측 최대 차이 / largest forecast gap {gap:.4f}  ->  {verdict}")
# 기대값 / Expected
#   Seasonal naive  평균 MAE / mean MAE : 0.395
#   A               평균 MAE / mean MAE : 0.059
#   B               평균 MAE / mean MAE : 0.066
#   C               평균 MAE / mean MAE : 0.069
#   21번의 적합 가운데 끝까지 가지 못한 것 / fits out of 21 that did not converge : [('B', '2018-09')]
#      B @ 2018-09: 로그가능도 / log-likelihood 369.00 -> 369.00, 예측 최대 차이 / largest forecast gap 0.0001  ->  표를 그대로 읽어도 됩니다 / the table stands
#   덱 오른쪽 표와 같습니다. 시작점을 일곱 번 바꾸면 세 후보는 훨씬 가까워집니다. 보류 한 번의 순서(A, B, C)를 믿지 않는 이유입니다.
#   The deck's right-hand table. Over seven starting points the three candidates are much closer, which is why we do not trust
#   the order of a single holdout.
#   끝까지 못 간 적합의 목록은 환경마다 다릅니다(이 노트북을 만든 Linux statsmodels 0.15.0: B 의 2018-09 하나, Linux 0.14.4: 없음).
#   그 적합을 다른 길로 다시 맞추면 로그가능도도 예측도 거의 그대로라(예측 차이 0.0001) 표는 바뀌지 않습니다. 5절 표 아래의 설명과 같은 이야기입니다.
#   The list of unfinished fits differs by environment (the Linux statsmodels 0.15.0 that built this notebook: one, B at 2018-09;
#   Linux 0.14.4: none). Refitting that one by another route leaves the log-likelihood and the forecast almost unchanged (forecast
#   gap 0.0001), so the table does not change. Same story as the note under the section 5 table.

In [ ]:
#@title ▶ Optional C. 3년 앞 예측과 2020년 / Three years ahead, and 2020 { display-mode: "form" }
fc = fits_full["C"].get_forecast(36)                        # 225개월 전체로 맞춘 C, 36개월 / candidate C fitted to all 225 months, 36 months
mean = fc.predicted_mean
low, high = fc.conf_int(alpha=0.05).T                       # 95% 예측 구간 / 95% prediction interval
future_months = pd.period_range("2019-10", periods=36, freq="M").to_timestamp()

# 그 뒤 실제로 일어난 일. 같은 계열(BLS CEU7000000001, 계절조정 전)의 2019-10 ~ 2022-09, 백만 명. 덱의 회색 점선과 같은 숫자.
# What actually happened next: the same series (BLS CEU7000000001, not seasonally adjusted), 2019-10 to 2022-09, millions. The deck's gray dashed line.
ACTUAL_2019_10_TO_2022_09 = [
    16.666, 16.459, 16.474, 16.117, 16.292, 15.745, 8.601, 10.208, 12.494, 13.173, 13.207, 13.161,
    13.273, 13.037, 12.467, 12.099, 12.540, 12.876, 13.435, 14.033, 14.800, 15.266, 15.239, 14.839,
    14.898, 14.861, 14.925, 14.592, 14.864, 15.103, 15.489, 15.933, 16.494, 16.703, 16.602, 16.143,
]
actual = np.array(ACTUAL_2019_10_TO_2022_09)

fig, ax = plt.subplots(figsize=(11, 3.6))
recent = jobs.loc["2014-01":]
ax.plot(recent.index.to_timestamp(), recent.to_numpy(), color=GREEN, label="observed")
ax.plot(future_months, mean, color=RED, label="mean forecast")
ax.fill_between(future_months, low, high, color=GOLD, alpha=0.35, label="95% interval")
ax.plot(future_months, actual, color="0.45", ls="--", label="actual (BLS)")
ax.set_title("candidate C, October 2019 to September 2022 (millions)")
ax.legend(loc="lower left")
plt.show()

width = high - low
inside = (low <= actual) & (actual <= high)                 # 달마다 실제 값이 띠 안인가 / is the actual value inside the band, month by month?
print("구간 폭 1 · 12 · 36개월 뒤 / interval width 1, 12, 36 months ahead :", np.round(width[[0, 11, 35]], 2))
print("띠 안에 든 달 / months inside the band :", int(inside.sum()), "of 36,  처음 띠 밖으로 나간 달 / first month outside :", future_months[np.argmin(inside)].strftime("%Y-%m"))
print("2020년 4월 / April 2020 : 실제 / actual", actual[6], " 띠의 아래 끝 / lower edge of the band", round(float(low[6]), 2))
drop = 1 - actual[6] / actual[5]                            # 2020년 3월 → 4월, 이 계열(계절조정 전)의 감소율 / the March-to-April 2020 drop on this (not seasonally adjusted) series
print("질문 / Question : 자료는 2019년 9월에 끝난다. 이 띠가 2020년을 담았을까? / Our data end in September 2019. Would this band have covered 2020?")
print(f"답 / Answer     : 아니다. 2020년 4월 한 달에 이 업종 일자리가 {drop:.0%} 줄었다(이 계열, 계절조정 전. BLS 발표의 계절조정 수치는 47%). 띠는 적합한 과거처럼 움직이는 미래만 담는다.")
print(f"                  No. These jobs fell {drop:.0%} in April 2020 alone (this series, not seasonally adjusted; BLS's seasonally adjusted figure is 47%). The band only covers a future that behaves like the fitted past.")
# 기대값 / Expected
#   구간 폭 1 · 12 · 36개월 뒤 / interval width 1, 12, 36 months ahead : [0.15 0.75 2.11]
#   띠 안에 든 달 / months inside the band : 5 of 36,  처음 띠 밖으로 나간 달 / first month outside : 2020-03
#   2020년 4월 / April 2020 : 실제 / actual 8.601  띠의 아래 끝 / lower edge of the band 16.59
#   질문 / Question : 자료는 2019년 9월에 끝난다. 이 띠가 2020년을 담았을까? / Our data end in September 2019. Would this band have covered 2020?
#   답 / Answer     : 아니다. 2020년 4월 한 달에 이 업종 일자리가 45% 줄었다(이 계열, 계절조정 전. BLS 발표의 계절조정 수치는 47%). 띠는 적합한 과거처럼 움직이는 미래만 담는다.
#                     No. These jobs fell 45% in April 2020 alone (this series, not seasonally adjusted; BLS's seasonally adjusted figure is 47%). The band only covers a future that behaves like the fitted past.
#   덱 「Three Years of Forecasts for the Jobs Series」 의 표와 같은 폭입니다. 띠는 지평과 함께 계속 넓어집니다(덱 「Why Do the Intervals Keep Widening?」).
#   The widths on the slide. The band keeps widening with the horizon ("Why Do the Intervals Keep Widening?").
#   회색 점선이 덱의 그림과 같습니다. 처음 다섯 달(2019-10 ~ 2020-02)은 띠 안, 2020-03 부터 2022-09 까지 31개월은 모두 띠 아래입니다.
#   The gray dashed line is the one on the slide. The first five months (2019-10 to 2020-02) sit inside the band; from 2020-03 to
#   2022-09 all 31 months fall below it.
#   모형이 틀려서가 아니라 자료를 만드는 방식이 바뀌었기 때문입니다(구조 변화). 어떤 모형의 띠도 이런 변화를 미리 담지 못합니다.
#   Not because the model was wrong, but because the process itself changed (a structural break). No model's band covers such
#   a change in advance.

In [ ]:
#@title ▶ Optional D. 시작점마다 후보 다시 고르기 / Choosing the candidates again at every starting point { display-mode: "form" }
# 선택 B 의 시작점마다, 그때까지의 자료로만 1 · 2 · 3절을 다시 돌린다: 결정표의 줄(원계열 → 계절 차분 뒤 → 두 차분 뒤)과 띠 밖의 시차.
# At each starting point of Optional B, rerun sections 1 to 3 on the data up to that point only: the decision-table rows
# (original -> after the seasonal difference -> after both) and the lags outside the band.
starts = ["2015-09", "2016-03", "2016-09", "2017-03", "2017-09", "2018-03", "2018-09"]
print("시작점 / start   줄 / rows: 원계열 ∇12 ∇∇12   ACF 띠 밖 / outside           PACF 띠 밖 / outside")
for s in starts:
    past = jobs.loc[:s]
    rows = []
    for x in (past, seasonal_difference(past, 12), seasonal_difference(seasonal_difference(past, 12), 1)):
        a, ap, k, kp = test_pair(x)
        rows.append(decision_row(ap, kp))
    x = seasonal_difference(seasonal_difference(past, 12), 1).dropna()
    band_s = 2 / np.sqrt(len(x))
    r1 = acf(x, nlags=36, fft=False)[1:]
    r2 = pacf(x, nlags=36, method="ywm")[1:]
    out1 = [k + 1 for k, v in enumerate(r1) if abs(v) > band_s]
    out2 = [k + 1 for k, v in enumerate(r2) if abs(v) > band_s]
    print(f"{s}          {rows[0]}    {rows[1]}    {rows[2]}       {str(out1):28s} {out2}")
# 기대값 / Expected
#   시작점 / start   줄 / rows: 원계열 ∇12 ∇∇12   ACF 띠 밖 / outside           PACF 띠 밖 / outside
#   2015-09          1    1    2       [1, 2, 11, 12, 22, 23, 36]   [1, 2, 12, 36]
#   2016-03          1    1    2       [1, 2, 11, 12, 22, 23, 36]   [1, 2, 12, 36]
#   2016-09          1    1    2       [1, 2, 11, 12, 22, 23]       [1, 2, 12, 36]
#   2017-03          1    1    2       [1, 2, 11, 12, 22, 23, 36]   [1, 2, 12, 36]
#   2017-09          1    1    2       [1, 2, 11, 12, 22, 23, 36]   [1, 2, 12, 36]
#   2018-03          1    1    2       [1, 2, 11, 12, 22, 23]       [1, 2, 12, 36]
#   2018-09          1    1    2       [1, 2, 11, 12, 22]           [1, 2, 11, 12]
#   시작점 일곱 번 모두 같은 읽기입니다. 줄은 1 → 1 → 2 (D = 1, d = 1), ACF 와 PACF 모두 시차 1 · 2 · 12 가 띠 밖이라
#   후보 A 와 B 를 그대로 씁니다. 11 · 22 · 23 · 36 처럼 띠를 겨우 넘는 막대는 시작점에 따라 들락거리고, 3절에서처럼 후보를 바꾸지 않습니다.
#   교재의 자동 탐색(statsforecast AutoARIMA)도 이 시작점들에서 매번 C 를 고릅니다(덱 노트에 기록, Colab 에 기본으로 없어 여기서는 돌리지 않음).
#   그래서 선택 B 가 후보 셋을 고정한 채 다시 맞춰도, 시작점마다 후보를 다시 골랐을 때와 같은 표가 됩니다.
#   All seven starting points read the same way: rows 1 -> 1 -> 2 (D = 1, d = 1), and lags 1, 2 and 12 are outside the band in
#   both plots, so candidates A and B stay as they are. Bars that just cross the band (11, 22, 23, 36) come and go with the
#   starting point and, as in section 3, do not change the candidates. The textbook's automatic search (statsforecast AutoARIMA)
#   also picks C at every one of these starting points (recorded in the deck notes; not preinstalled in Colab, so not run here).
#   So refitting the fixed three in Optional B gives the same table as choosing the candidates again at every starting point.

---
### 이번 주에 가져갈 것 / What to take away

덱 「Today in Three Sentences」 와 같은 내용을 이 노트북의 절로 다시 적었습니다.
*The same points as "Today in Three Sentences", tied to the sections of this notebook.*

1. 차분은 검정과 그림이 말하는 만큼만 합니다(1 · 2절). 검정은 「아직 정상이 아니다」 까지만 말하고, 어떤 차분을 먼저 시도할지는 그림이 제안하며, 시도한 결과는 검정 · 잔차 · 예측으로 점검합니다. 직선을 따라 오르는 계열은 추세판도 봅니다. 한 번 더 하면 시차 1 에 −0.5 근처의 막대가 생기기 쉽습니다. 그 막대는 증명이 아니라 경고라서, 그 차분이 필요했는지 다시 봅니다.
   *Difference only as many times as the tests and the plot say (sections 1, 2). The tests only say "not stationary yet"; the plot suggests which difference to try first, and the tests, residuals and forecasts check the result. For a series rising along a line, also run the trend versions. One difference too many tends to leave a bar near −0.5 at lag 1. That bar is a warning, not proof, so go back and check whether the difference was needed.*
2. 차분한 뒤에는 6주차처럼 ACF 와 PACF 를 읽어 SARIMA 후보 몇 개를 씁니다(3절). 짧은 시차는 $p, q$, 12 의 배수는 $P, Q$.
   *After differencing, read the ACF and PACF as in Week 6 and write a few SARIMA candidates (section 3): short lags for $p, q$, multiples of 12 for $P, Q$.*
3. 잔차가 잡음처럼 보이고(4절) 나중 자료에서 단순한 예측을 이기는(5절) 모형만 남깁니다. 후보 선택과 적합에는 시험 구간을 쓰지 않습니다.
   *Keep a model only if its residuals look like noise (section 4) and it beats a simple forecast on later data (section 5). Neither choosing nor fitting a candidate may use the test months.*

**Your turn.** 선택 하나를 바꾸고(예: 3절의 후보, 또는 5절 첫 셀의 `TRAIN_END = "2017-09"` 를 `"2018-09"` 로 바꿔 시험을 12개월로), 5절을 다시 돌려 결과가 어떻게 달라졌는지 한 문장으로 적어 보세요. 덱 「In the Lab, Run All Five Steps on One Series」 의 「Change one choice, explain one difference」.
*Change one choice (for example a candidate in section 3, or set `TRAIN_END = "2018-09"` in the first cell of section 5 for a 12-month test), rerun section 5, and explain the difference in one sentence: "Change one choice, explain one difference".*

개인 오류는 코드 셀 번호와 마지막 오류 메시지를 함께 수업 게시판에 남기세요. 마지막 셀은 실행 환경을 적는 기록용입니다.
*For an error, post the code cell number and the last error message on the course board. The last cell records the environment.*

References: [FPP-py §9.1 Stationarity and differencing](https://otexts.com/fpppy/nbs/09-arima.html) · [FPP-py §9.9 Seasonal ARIMA](https://otexts.com/fpppy/nbs/09-arima.html) · [statsmodels SARIMAX](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.sarimax.SARIMAX.html)

In [ ]:
# 연습 공간 / Scratch space

In [ ]:
# 실행 환경 기록. 오류를 게시판에 남길 때 이 출력도 함께 붙여 주세요.
# A record of the environment. Paste this output too when you post an error on the board.
import sys, platform, datetime, matplotlib
print("date        :", datetime.datetime.now().isoformat(timespec="seconds"))
print("python      :", sys.version.split()[0], "on", platform.platform())
print("numpy       :", np.__version__, "· pandas", pd.__version__)
print("statsmodels :", statsmodels.__version__, "· matplotlib", matplotlib.__version__)
print("random      : seed 101 (section 1, Week 5 series A and B) only")